# 05 — 09:30→09:35 因子深度研究

## 研究目标

使用 09:15–09:25 集合竞价信息及严格 09:30 时点可见盘口，研究其与
09:30→09:35 个股原始收益率的横截面相关性。

本 notebook 不以复杂模型点预测为主，而以以下指标评价因子：

- 日度 Spearman Rank IC；
- ICIR 与正 IC 日期比例；
- Q5−Q1 分组收益及其稳定性；
- 五组收益单调性；
- 日期、板块和流动性分层稳健性。

“机构资金”只能表述为 `institution-like order-flow proxy`，不能根据
Level-2 委托规模直接认定真实账户身份。


## 执行结构

1. 数据和严格标签；
2. 竞价因子与严格 09:30 盘口合并；
3. 数据质量和时间可得性；
4. 基础因子统一排名；
5. 因子树 A：严格 09:30 盘口压力；
6. 因子树 B：竞价缺口、吸收与反转；
7. 因子树 C：大单与参与者结构；
8. 因子裂变的日期/流动性稳健性；
9. 少量有效因子的机械组合；
10. 未来日期的一次性验证。

原 notebook 中的 tail classifier、direction/magnitude components、
交易成本和延长持有期研究保留在原文件中，不进入本 notebook 主线。


In [1]:
from dataclasses import dataclass
from typing import Dict, Iterable, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

EPS = 1e-9


def safe_divide(a, b):
    """Element-wise division that returns NaN when the denominator is too small."""
    return np.where(np.abs(b) > EPS, a / b, np.nan)


def require_columns(df: pd.DataFrame, columns: Sequence[str], name: str = "DataFrame") -> None:
    """Fail early with an informative error when required columns are missing."""
    missing = [col for col in columns if col not in df.columns]
    if missing:
        raise KeyError(f"{name} is missing required columns: {missing}")


def cross_sectional_winsorize(
    df: pd.DataFrame,
    columns: Sequence[str],
    group_cols: Sequence[str] = ("date", "minute"),
    lower_q: float = 0.01,
    upper_q: float = 0.99,
) -> pd.DataFrame:
    """Winsorize features within each date-minute cross-section without using future data."""
    result = df.copy()

    for col in columns:
        if col not in result.columns:
            continue

        lower = result.groupby(list(group_cols))[col].transform(
            lambda x: x.quantile(lower_q)
        )
        upper = result.groupby(list(group_cols))[col].transform(
            lambda x: x.quantile(upper_q)
        )
        result[col] = result[col].clip(lower=lower, upper=upper)

    return result


In [5]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from src.ddb_client import connect_ddb

session = connect_ddb()


In [3]:
@dataclass(frozen=True)
class Config:
    # Data source
    DB_PATH: str = "dfs://quota"

    # QUICK_MODE preserves the original short sample for fast iteration.
    # Set QUICK_MODE=False for the extended research run.
    QUICK_MODE: bool = False

    QUICK_START_DATE: str = "2026.03.11"
    RESEARCH_START_DATE: str = "2026.01.01"
    END_DATE: str = "2026.04.10"

    START_TIME: str = "09:29:45"
    END_TIME: str = "09:36:00"

    QUICK_N_STOCKS: int = 30
    RESEARCH_N_STOCKS: int = 40

    RANDOM_STATE: int = 42
    HORIZON_MINUTES: int = 5
    ROLLING_WINDOW: int = 20

    RF_PARAMS: Optional[Dict] = None
    XGB_PARAMS: Optional[Dict] = None

    @property
    def START_DATE(self) -> str:
        return self.QUICK_START_DATE if self.QUICK_MODE else self.RESEARCH_START_DATE

    @property
    def N_STOCKS(self) -> int:
        return self.QUICK_N_STOCKS if self.QUICK_MODE else self.RESEARCH_N_STOCKS


CFG = Config(
    RF_PARAMS={
        "n_estimators": 200,
        "max_depth": 5,
        "min_samples_leaf": 50,
        "max_features": "sqrt",
        "random_state": 42,
        "n_jobs": -1,
    },
    XGB_PARAMS={
        "n_estimators": 300,
        "max_depth": 3,
        "learning_rate": 0.03,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "objective": "reg:squarederror",
        "random_state": 42,
        "n_jobs": -1,
    },
)

DB_PATH = CFG.DB_PATH
START_DATE = CFG.START_DATE
END_DATE = CFG.END_DATE
START_TIME = CFG.START_TIME
END_TIME = CFG.END_TIME
RANDOM_STATE = CFG.RANDOM_STATE
N_STOCKS = CFG.N_STOCKS
ROLLING_WINDOW = CFG.ROLLING_WINDOW
HORIZON_MINUTES = CFG.HORIZON_MINUTES

print("Mode:", "QUICK" if CFG.QUICK_MODE else "EXTENDED RESEARCH")
print("Database:", DB_PATH)
print("Sample period:", START_DATE, "to", END_DATE)
print("Intraday window:", START_TIME, "to", END_TIME)
print("N_STOCKS:", N_STOCKS)
print("Rolling window:", ROLLING_WINDOW, "trading days")
print("Prediction horizon:", HORIZON_MINUTES, "minutes")


Mode: EXTENDED RESEARCH
Database: dfs://quota
Sample period: 2026.01.01 to 2026.04.10
Intraday window: 09:29:45 to 09:36:00
N_STOCKS: 40
Rolling window: 20 trading days
Prediction horizon: 5 minutes


## 2. Unified Research Sample

File 4 and File 5 use the same deterministic 30-stock universe. The filter combines `qtick.dtype == 1`, exchange-aware A-share code ranges, at least 20 covered trading dates, and explicit exclusion checks for common index, ETF, fund and convertible-bond ranges. The final pool is stratified across SZ Main, SH Main, ChiNext and STAR so that the later board, price-group, liquidity and cross-sectional prediction analyses refer to the same underlying stocks.


In [6]:
# ============================================================
# Unified A-share Stock Universe (shared by File 4 and File 5)
# ============================================================
# qtick has no separate security master. We therefore use two independent
# filters: (1) dtype == 1 from qtick, and (2) an exchange-aware A-share code
# whitelist. A prefix alone is unsafe: e.g. 000170.SH is an index, not SZ Main.

POOL_START_DATE = START_DATE
POOL_END_DATE = END_DATE
POOL_RANDOM_STATE = RANDOM_STATE
MIN_QTICK_DATES = 55

BOARD_SAMPLE_PLAN = {
    "SZ Main": 12,
    "SH Main": 12,
    "ChiNext": 8,
    "STAR": 8,
}

universe_script = f"""
qtick = loadTable("{DB_PATH}", "qtick")

select distinct code, date, dtype, name
from qtick
where date >= {POOL_START_DATE}
  and date <= {POOL_END_DATE}
  and dtype = 1
  and (
        code like "000%.SZ"
        or code like "001%.SZ"
        or code like "002%.SZ"
        or code like "003%.SZ"
        or code like "300%.SZ"
        or code like "301%.SZ"
        or code like "600%.SH"
        or code like "601%.SH"
        or code like "603%.SH"
        or code like "605%.SH"
        or code like "688%.SH"
      )
order by code, date
"""

universe_daily = session.run(universe_script)
universe_daily["code"] = (
    universe_daily["code"].astype(str).str.strip().str.upper()
)


def classify_a_share_board(code):
    """Exchange-aware whitelist for ordinary Shanghai/Shenzhen A shares."""
    code = str(code).strip().upper()
    if code.endswith(".SZ") and code[:3] in {"000", "001", "002", "003"}:
        return "SZ Main"
    if code.endswith(".SZ") and code[:3] in {"300", "301"}:
        return "ChiNext"
    if code.endswith(".SH") and code[:3] in {"600", "601", "603", "605"}:
        return "SH Main"
    if code.endswith(".SH") and code[:3] == "688":
        return "STAR"
    return None


universe_daily["board"] = universe_daily["code"].map(classify_a_share_board)

stock_coverage = (
    universe_daily
    .groupby(["code", "board"], as_index=False)
    .agg(
        name=("name", "first"),
        dtype=("dtype", "first"),
        n_qtick_dates=("date", "nunique"),
    )
)

eligible_pool = stock_coverage[
    stock_coverage["board"].notna()
    & stock_coverage["dtype"].eq(1)
    & stock_coverage["n_qtick_dates"].ge(MIN_QTICK_DATES)
].copy()

selected_parts = []
for board, n_target in BOARD_SAMPLE_PLAN.items():
    board_pool = eligible_pool[eligible_pool["board"].eq(board)].sort_values("code")
    if len(board_pool) < n_target:
        raise ValueError(
            f"Insufficient eligible {board} stocks: need {n_target}, found {len(board_pool)}"
        )
    selected_parts.append(
        board_pool.sample(n=n_target, random_state=POOL_RANDOM_STATE)
    )

stock_pool = (
    pd.concat(selected_parts, ignore_index=True)
    .sort_values(["board", "code"])
    .reset_index(drop=True)
)
sample_codes = stock_pool["code"].tolist()

# Fail loudly if a non-equity code can enter the modelling sample.
assert stock_pool["code"].is_unique
assert stock_pool["dtype"].eq(1).all()
assert stock_pool["board"].notna().all()
assert stock_pool["code"].map(classify_a_share_board).notna().all()
assert not stock_pool["code"].isin(["000001.SH", "399001.SZ", "399006.SZ"]).any()
assert not stock_pool["code"].str.startswith(
    ("159", "510", "511", "512", "513", "515", "516", "518", "588",
     "110", "113", "123", "127", "128")
).any()

codes_ddb = "[" + ",".join(f'"{code}"' for code in sample_codes) + "]"
code_filter = ",".join(f'"{code}"' for code in sample_codes)

print("Eligible ordinary A-share stocks:", len(eligible_pool))
print("Unified selected stocks:", len(sample_codes))
display(stock_pool.groupby("board").size().rename("n_stocks").to_frame())
display(stock_pool[["code", "name", "dtype", "board", "n_qtick_dates"]])



# ============================================================
# Three-table coverage validation for the unified stock pool
# ============================================================
coverage_queries = {
    "qtick": f"""
        select distinct code, date
        from loadTable("{DB_PATH}", "qtick")
        where code in {codes_ddb}
          and date >= {POOL_START_DATE}
          and date <= {POOL_END_DATE}
    """,
    "qorder": f"""
        select distinct code, date
        from loadTable("{DB_PATH}", "qorder")
        where code in {codes_ddb}
          and date >= {POOL_START_DATE}
          and date <= {POOL_END_DATE}
    """,
    "qknock": f"""
        select distinct code, date
        from loadTable("{DB_PATH}", "qknock")
        where code in {codes_ddb}
          and date >= {POOL_START_DATE}
          and date <= {POOL_END_DATE}
    """,
}

coverage_parts = []
for table_name, query in coverage_queries.items():
    tmp = session.run(query)
    tmp["code"] = tmp["code"].astype(str).str.strip().str.upper()
    summary = tmp.groupby("code")["date"].nunique().rename(f"{table_name}_dates")
    coverage_parts.append(summary)

three_table_coverage = (
    stock_pool.set_index("code")[["name", "board"]]
    .join(coverage_parts, how="left")
    .fillna({"qtick_dates": 0, "qorder_dates": 0, "qknock_dates": 0})
    .reset_index()
)

for col in ["qtick_dates", "qorder_dates", "qknock_dates"]:
    three_table_coverage[col] = three_table_coverage[col].astype(int)

three_table_coverage["three_table_ready"] = (
    three_table_coverage[["qtick_dates", "qorder_dates", "qknock_dates"]]
    .min(axis=1)
    .ge(MIN_QTICK_DATES)
)

display(three_table_coverage)
assert three_table_coverage["three_table_ready"].all(), (
    "Some selected stocks lack sufficient qtick/qorder/qknock coverage. "
    "Inspect three_table_coverage before continuing."
)
print("Unified stock pool passed security-type, code-range and three-table coverage checks.")


Eligible ordinary A-share stocks: 5180
Unified selected stocks: 40


,n_stocks
board,
ChiNext,8
SH Main,12
STAR,8
SZ Main,12


,code,name,dtype,board,n_qtick_dates
0,300183.SZ,东软载波,1,ChiNext,63
1,300490.SZ,华自科技,1,ChiNext,63
2,300595.SZ,欧普康视,1,ChiNext,63
3,301052.SZ,果麦文化,1,ChiNext,63
4,301255.SZ,通力科技,1,ChiNext,63
5,301391.SZ,卡莱特,1,ChiNext,63
6,301552.SZ,科力装备,1,ChiNext,63
7,301622.SZ,英思特,1,ChiNext,63
8,600158.SH,中体产业,1,SH Main,63
9,600182.SH,S佳通,1,SH Main,63


,code,name,board,qtick_dates,qorder_dates,qknock_dates,three_table_ready
0,300183.SZ,东软载波,ChiNext,63,63,63,True
1,300490.SZ,华自科技,ChiNext,63,63,63,True
2,300595.SZ,欧普康视,ChiNext,63,63,63,True
3,301052.SZ,果麦文化,ChiNext,63,63,63,True
4,301255.SZ,通力科技,ChiNext,63,63,63,True
5,301391.SZ,卡莱特,ChiNext,63,63,63,True
6,301552.SZ,科力装备,ChiNext,63,63,63,True
7,301622.SZ,英思特,ChiNext,63,63,63,True
8,600158.SH,中体产业,SH Main,63,63,63,True
9,600182.SH,S佳通,SH Main,63,63,63,True


Unified stock pool passed security-type, code-range and three-table coverage checks.


## 3. Load `qtick` Snapshot Data

The first version builds a 1-minute panel from `qtick` snapshot data. Only necessary fields are queried to reduce DolphinDB transfer size.


In [7]:
# ============================================================
# Batched qtick Query for File 5
# ============================================================

qtick_cols = """
code, date, time,
new_price, pre_close,
bp0, ap0,
bv0, av0,
bv1, av1,
bv2, av2,
bv3, av3,
bv4, av4,
sum_volume, sum_amount
"""


def chunk_codes(codes, batch_size=10):
    for start in range(0, len(codes), batch_size):
        yield codes[start:start + batch_size]


qtick_parts = []

for batch_no, code_batch in enumerate(
    chunk_codes(sample_codes, batch_size=10),
    start=1
):
    batch_codes_ddb = (
        "["
        + ",".join(
            f'"{code}"'
            for code in code_batch
        )
        + "]"
    )

    qtick_script = f"""
    select {qtick_cols}
    from loadTable("{DB_PATH}", "qtick")
    where code in {batch_codes_ddb}
      and date >= {START_DATE}
      and date <= {END_DATE}
      and time >= {START_TIME}.000
      and time <= {END_TIME}.000
    order by code, date, time
    """

    batch_df = session.run(qtick_script)
    qtick_parts.append(batch_df)

    print(
        f"File 5 qtick batch {batch_no}:",
        len(code_batch),
        "stocks,",
        len(batch_df),
        "rows"
    )

qtick = pd.concat(
    qtick_parts,
    ignore_index=True
)

print("qtick total shape:", qtick.shape)
display(qtick.head())

File 5 qtick batch 1: 10 stocks, 73520 rows
File 5 qtick batch 2: 10 stocks, 75341 rows
File 5 qtick batch 3: 10 stocks, 74401 rows
File 5 qtick batch 4: 10 stocks, 76007 rows
qtick total shape: (299269, 19)


,code,date,time,new_price,pre_close,bp0,ap0,bv0,av0,bv1,av1,bv2,av2,bv3,av3,bv4,av4,sum_volume,sum_amount
0,300183.SZ,2026-01-05,1970-01-01 09:30:00,14.96,14.83,14.92,14.96,100,3100,2000,10000,100,1000,300,3200,1400,200,104800,1567017.0
1,300183.SZ,2026-01-05,1970-01-01 09:30:03,14.97,14.83,14.97,14.99,4900,10000,21400,15000,43000,5100,400,500,2700,2800,155400,2325004.0
2,300183.SZ,2026-01-05,1970-01-01 09:30:06,14.96,14.83,14.96,14.99,21700,11800,46900,15000,400,5100,3200,500,2500,2800,164500,2461201.0
3,300183.SZ,2026-01-05,1970-01-01 09:30:09,14.99,14.83,14.98,14.99,1100,8000,22100,15000,46900,5100,3500,2200,3000,500,169400,2534630.0
4,300183.SZ,2026-01-05,1970-01-01 09:30:12,14.98,14.83,14.96,14.98,22100,1300,46900,8000,3500,15000,3000,5100,1500,2200,170800,2555602.0


## 4. Minute-Level Market State

Each final row represents one stock-minute:

```text
stock × date × minute
```

The row stores the last observable snapshot within that minute.


In [8]:
qtick = qtick.copy()

date_str = pd.to_datetime(qtick["date"]).dt.strftime("%Y-%m-%d")

time_str = qtick["time"].astype(str)
time_str = (
    time_str
    .str.replace("1970-01-01 ", "", regex=False)
    .str.replace("1900-01-01 ", "", regex=False)
)

qtick["datetime"] = pd.to_datetime(
    date_str + " " + time_str,
    format="mixed"
)

qtick["time_clean"] = qtick["datetime"].dt.strftime("%H:%M:%S.%f")
qtick["time_clean"] = qtick["time_clean"].str.replace(".000000", "", regex=False)

qtick["minute"] = qtick["datetime"].dt.floor("min")

display(qtick[["code", "date", "time_clean", "datetime", "minute"]].head())


,code,date,time_clean,datetime,minute
0,300183.SZ,2026-01-05,09:30:00,2026-01-05 09:30:00,2026-01-05 09:30:00
1,300183.SZ,2026-01-05,09:30:03,2026-01-05 09:30:03,2026-01-05 09:30:00
2,300183.SZ,2026-01-05,09:30:06,2026-01-05 09:30:06,2026-01-05 09:30:00
3,300183.SZ,2026-01-05,09:30:09,2026-01-05 09:30:09,2026-01-05 09:30:00
4,300183.SZ,2026-01-05,09:30:12,2026-01-05 09:30:12,2026-01-05 09:30:00


In [9]:
qtick["bp0"] = pd.to_numeric(qtick["bp0"], errors="coerce").replace(0, np.nan)
qtick["ap0"] = pd.to_numeric(qtick["ap0"], errors="coerce").replace(0, np.nan)

qtick["mid_price"] = (qtick["bp0"] + qtick["ap0"]) / 2

display(qtick[["code", "date", "datetime", "bp0", "ap0", "mid_price"]].head())


,code,date,datetime,bp0,ap0,mid_price
0,300183.SZ,2026-01-05,2026-01-05 09:30:00,14.92,14.96,14.940
1,300183.SZ,2026-01-05,2026-01-05 09:30:03,14.97,14.99,14.980
2,300183.SZ,2026-01-05,2026-01-05 09:30:06,14.96,14.99,14.975
3,300183.SZ,2026-01-05,2026-01-05 09:30:09,14.98,14.99,14.985
4,300183.SZ,2026-01-05,2026-01-05 09:30:12,14.96,14.98,14.970


In [10]:
minute_panel = (
    qtick
    .sort_values(["code", "date", "datetime"])
    .groupby(["code", "date", "minute"])
    .agg(
        mid_price=("mid_price", "last"),
        last_price=("new_price", "last"),
        pre_close=("pre_close", "last"),
        bp0=("bp0", "last"),
        ap0=("ap0", "last"),
        bv0=("bv0", "last"),
        av0=("av0", "last"),
        bv1=("bv1", "last"),
        av1=("av1", "last"),
        bv2=("bv2", "last"),
        av2=("av2", "last"),
        bv3=("bv3", "last"),
        av3=("av3", "last"),
        bv4=("bv4", "last"),
        av4=("av4", "last"),
        sum_volume=("sum_volume", "last"),
        sum_amount=("sum_amount", "last")
    )
    .reset_index()
)

minute_panel = minute_panel.sort_values(["code", "date", "minute"])

print("minute_panel shape:", minute_panel.shape)
display(minute_panel.head())


minute_panel shape: (16768, 20)


,code,date,minute,mid_price,last_price,pre_close,bp0,ap0,bv0,av0,bv1,av1,bv2,av2,bv3,av3,bv4,av4,sum_volume,sum_amount
0,000713.SZ,2026-01-05,2026-01-05 09:30:00,6.625,6.63,6.62,6.62,6.63,57900,42400,97700,51200,85600,63400,20200,112600,59100,93000,222800,1478169.0
1,000713.SZ,2026-01-05,2026-01-05 09:31:00,6.645,6.64,6.62,6.64,6.65,200,73900,100100,73000,60400,111600,98700,98000,88900,74700,394100,2615055.0
2,000713.SZ,2026-01-05,2026-01-05 09:32:00,6.615,6.62,6.62,6.61,6.62,107700,6300,95400,28000,20200,47700,58700,134900,13300,82100,571100,3788036.0
3,000713.SZ,2026-01-05,2026-01-05 09:33:00,6.625,6.63,6.62,6.62,6.63,73800,40900,114700,83600,95600,134300,21700,81200,68700,114600,614800,4077336.0
4,000713.SZ,2026-01-05,2026-01-05 09:34:00,6.625,6.63,6.62,6.62,6.63,153900,19700,128300,86100,97200,134300,21700,81200,70700,115300,652200,4325279.0


## 5. Exact-Horizon Label Construction

The target is the midpoint return from timestamp `t` to the exact timestamp `t + 5 minutes`.

```text
future_return_5m = mid_price(t + 5min) / mid_price(t) - 1
```

A row-based `.shift(-5)` can silently misalign the label when one or more minutes are missing. The corrected implementation creates an explicit target timestamp and performs an exact `code × date × target_minute` match.

Observations without an exact `t+5min` midpoint are retained in the minute panel for diagnostics but excluded from the labeled modeling sample.


In [11]:
# ============================================================
# Exact-Horizon Label Construction
# ============================================================

require_columns(
    minute_panel,
    ["code", "date", "minute", "mid_price"],
    name="minute_panel",
)

minute_panel = (
    minute_panel
    .sort_values(["code", "date", "minute"])
    .copy()
)

# Exact target timestamp rather than row-based shift(-5).
minute_panel["target_minute_5m"] = (
    minute_panel["minute"]
    + pd.Timedelta(minutes=HORIZON_MINUTES)
)

future_price_lookup = (
    minute_panel[
        ["code", "date", "minute", "mid_price"]
    ]
    .rename(
        columns={
            "minute": "target_minute_5m",
            "mid_price": "future_mid_5m",
        }
    )
    .drop_duplicates(
        subset=["code", "date", "target_minute_5m"]
    )
)

minute_panel = minute_panel.merge(
    future_price_lookup,
    on=["code", "date", "target_minute_5m"],
    how="left",
    validate="many_to_one",
)

minute_panel["future_return_5m"] = (
    minute_panel["future_mid_5m"]
    / minute_panel["mid_price"]
    - 1
)

minute_panel["future_direction_5m"] = np.select(
    [
        minute_panel["future_return_5m"] > 0,
        minute_panel["future_return_5m"] < 0,
    ],
    [1, -1],
    default=0,
)

minute_panel["exact_horizon_available"] = (
    minute_panel["future_mid_5m"].notna()
)

label_df = minute_panel.dropna(
    subset=["mid_price", "future_mid_5m", "future_return_5m"]
).copy()

label_quality = pd.DataFrame({
    "Metric": [
        "Minute-panel observations",
        "Exact 5-minute labels",
        "Missing exact-horizon ratio",
        "Unique dates",
        "Unique stocks",
    ],
    "Value": [
        len(minute_panel),
        len(label_df),
        1 - minute_panel["exact_horizon_available"].mean(),
        label_df["date"].nunique(),
        label_df["code"].nunique(),
    ],
})

display(label_quality)

display(
    label_df[
        [
            "code",
            "date",
            "minute",
            "target_minute_5m",
            "mid_price",
            "future_mid_5m",
            "future_return_5m",
            "future_direction_5m",
        ]
    ].head(10)
)


,Metric,Value
0,Minute-panel observations,16768.000000
1,Exact 5-minute labels,4146.000000
2,Missing exact-horizon ratio,0.752684
3,Unique dates,63.000000
4,Unique stocks,40.000000


,code,date,minute,target_minute_5m,mid_price,future_mid_5m,future_return_5m,future_direction_5m
0,000713.SZ,2026-01-05,2026-01-05 09:30:00,2026-01-05 09:35:00,6.625,6.635,0.001509,1
1,000713.SZ,2026-01-05,2026-01-05 09:31:00,2026-01-05 09:36:00,6.645,6.625,-0.003010,-1
7,000713.SZ,2026-01-06,2026-01-06 09:30:00,2026-01-06 09:35:00,6.675,6.715,0.005993,1
8,000713.SZ,2026-01-06,2026-01-06 09:31:00,2026-01-06 09:36:00,6.675,6.715,0.005993,1
14,000713.SZ,2026-01-07,2026-01-07 09:30:00,2026-01-07 09:35:00,6.745,6.745,0.000000,0
20,000713.SZ,2026-01-08,2026-01-08 09:30:00,2026-01-08 09:35:00,6.725,6.755,0.004461,1
21,000713.SZ,2026-01-08,2026-01-08 09:31:00,2026-01-08 09:36:00,6.755,6.755,0.000000,0
27,000713.SZ,2026-01-09,2026-01-09 09:30:00,2026-01-09 09:35:00,6.745,6.765,0.002965,1
28,000713.SZ,2026-01-09,2026-01-09 09:31:00,2026-01-09 09:36:00,6.755,6.765,0.001480,1
34,000713.SZ,2026-01-12,2026-01-12 09:30:00,2026-01-12 09:35:00,6.765,6.755,-0.001478,-1


In [12]:
# ============================================================
# Exact 5-Minute Horizon Check
# ============================================================

minute_panel["future_minute_5m"] = (
    minute_panel
    .groupby(["code", "date"])["minute"]
    .shift(-5)
)

minute_panel["actual_horizon_min"] = (
    (minute_panel["future_minute_5m"] - minute_panel["minute"])
    .dt.total_seconds() / 60
)

display(
    minute_panel["actual_horizon_min"]
    .value_counts(dropna=False)
    .sort_index()
    .rename("count")
    .to_frame()
)


,count
actual_horizon_min,
5.0,4168
NaN,12600


# Part II. 严格开盘样本与集合竞价特征

以下单元复用原研究中已经验证的数据合并逻辑。Notebook 04 仍是竞价
事件级因子的生产端，本 notebook 只负责标签、评价和机制比较。


In [19]:
# ============================================================
# 21.2 Construct Strict 09:30 Opening Sample
# ============================================================

OPENING_TIME = "09:30:00"

opening_feature_candidates = [
    "relative_spread",
    "log_top_depth",
    "depth_imbalance",
    "microprice_deviation"
]

opening_features = [
    feature
    for feature in opening_feature_candidates
    if feature in master_feature_df.columns
]

missing_opening_features = [
    feature
    for feature in opening_feature_candidates
    if feature not in master_feature_df.columns
]

print(
    "Opening features:",
    opening_features
)

print(
    "Missing opening features:",
    missing_opening_features
)

opening_model_df = (
    master_feature_df
    .copy()
)

opening_model_df["date"] = pd.to_datetime(
    opening_model_df["date"]
)

opening_model_df["minute"] = pd.to_datetime(
    opening_model_df["minute"]
)

opening_model_df = (
    opening_model_df[
        opening_model_df["minute"]
        .dt.strftime("%H:%M:%S")
        .eq(OPENING_TIME)
    ]
    .dropna(
        subset=(
            opening_features
            + [
                "future_return_5m",
                "mid_price"
            ]
        )
    )
    .copy()
)

opening_model_df[
    "future_return_rank"
] = (
    opening_model_df
    .groupby(
        "date"
    )["future_return_5m"]
    .rank(
        pct=True,
        method="average"
    )
)

print(
    "Opening observations:",
    len(opening_model_df)
)

print(
    "Opening dates:",
    opening_model_df["date"].nunique()
)

print(
    "Opening stocks:",
    opening_model_df["code"].nunique()
)

display(
    opening_model_df[
        [
            "date",
            "minute",
            "code",
            "mid_price"
        ]
        + opening_features
        + [
            "future_return_5m",
            "future_return_rank"
        ]
    ].head()
)

Opening features: ['relative_spread', 'log_top_depth', 'depth_imbalance', 'microprice_deviation']
Missing opening features: []
Opening observations: 2506
Opening dates: 63
Opening stocks: 40


,date,minute,code,mid_price,relative_spread,log_top_depth,depth_imbalance,microprice_deviation,future_return_5m,future_return_rank
0,2026-01-05,2026-01-05 09:30:00,000713.SZ,6.625,0.001509,11.515931,0.154536,0.000117,0.001509,0.525000
2,2026-01-06,2026-01-06 09:30:00,000713.SZ,6.675,0.001498,12.163170,-0.219207,-0.000164,0.005993,0.625000
4,2026-01-07,2026-01-07 09:30:00,000713.SZ,6.745,0.001483,10.034560,0.508772,0.000377,0.000000,0.435897
5,2026-01-08,2026-01-08 09:30:00,000713.SZ,6.725,0.001487,10.138599,0.976285,0.000726,0.004461,0.512821
7,2026-01-09,2026-01-09 09:30:00,000713.SZ,6.745,0.001483,11.388960,-0.760018,-0.000563,0.002965,0.475000


In [20]:
# ============================================================
# 22.1 Load File 4 Auction Feature Table
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

AUCTION_FEATURE_CANDIDATES = [
    # 优先读取File4最后导出的完整表
    Path("04_auction_feature_table.csv"),
    Path("../notebooks/04_auction_feature_table.csv"),

    # 旧版本只作为fallback
    Path("../notebooks/auction_features_prediction_ready.csv"),

    Path(
        r"C:\Users\work\OneDrive\Documents"
        r"\level2-research\notebooks\04_auction_feature_table.csv"
    ),
]

AUCTION_FEATURE_PATH = next(
    (path for path in AUCTION_FEATURE_CANDIDATES if path.exists()),
    None,
)
if AUCTION_FEATURE_PATH is None:
    raise FileNotFoundError(
        "Run the organized File 4 export first, then update AUCTION_FEATURE_CANDIDATES."
    )

auction_feature_df = pd.read_csv(AUCTION_FEATURE_PATH)

# Standardize primary keys
auction_feature_df["code"] = (
    auction_feature_df["code"]
    .astype(str)
    .str.strip()
    .str.upper()
)

auction_feature_df["date"] = (
    pd.to_datetime(
        auction_feature_df["date"],
        errors="coerce"
    )
    .dt.normalize()
)

# Columns that should not be forcibly converted to numeric
NON_NUMERIC_COLUMNS = {
    "code",
    "date",
    "auction_price_consistent"
}

# Define numeric feature columns directly from the CSV
auction_numeric_cols = [
    column
    for column in auction_feature_df.columns
    if column not in NON_NUMERIC_COLUMNS
]

# Convert File 4 features to numeric
auction_feature_df[auction_numeric_cols] = (
    auction_feature_df[auction_numeric_cols]
    .apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
)

# Convert boolean consistency flag separately
if "auction_price_consistent" in auction_feature_df.columns:
    auction_feature_df["auction_price_consistent"] = (
        auction_feature_df["auction_price_consistent"]
        .astype(str)
        .str.lower()
        .map({
            "true": 1,
            "false": 0,
            "1": 1,
            "0": 0
        })
    )

# Primary-key validation
duplicate_keys = auction_feature_df.duplicated(
    ["code", "date"]
).sum()

print("Auction feature shape:", auction_feature_df.shape)
print("Auction stocks:", auction_feature_df["code"].nunique())
print("Auction dates:", auction_feature_df["date"].nunique())
print("Duplicate code-date rows:", duplicate_keys)

assert duplicate_keys == 0, (
    "File 4 contains duplicate code-date rows."
)

# Required fields for boss-directed analysis
required_auction_columns = [
    "code",
    "date",
    "auction_price",
    "auction_trade_volume",
    "auction_trade_amount",
    "prev_day_return"
]

missing_required_columns = [
    column
    for column in required_auction_columns
    if column not in auction_feature_df.columns
]

assert not missing_required_columns, (
    "Missing required File 4 columns: "
    f"{missing_required_columns}"
)

# Price-scale check
auction_price_summary = (
    auction_feature_df["auction_price"]
    .describe()
    .to_frame("auction_price")
)

display(auction_price_summary)

median_auction_price = (
    auction_feature_df["auction_price"]
    .median()
)

assert median_auction_price < 1000, (
    "auction_price appears to remain in raw QKNOCK units."
)

preview_columns = [
    "code",
    "date",
    "auction_price",
    "auction_trade_volume",
    "auction_trade_amount",
    "prev_day_return",
    "has_auction_execution",
    "large_order_amount_share",
    "large_order_imbalance",
    "post920_large_order_imbalance"
]

preview_columns = [
    column
    for column in preview_columns
    if column in auction_feature_df.columns
]

print("Loaded file:", AUCTION_FEATURE_PATH.resolve())
print("Loaded columns:", len(auction_feature_df.columns))

display(
    auction_feature_df[
        preview_columns
    ].head(10)
)

Auction feature shape: (2520, 81)
Auction stocks: 40
Auction dates: 63
Duplicate code-date rows: 0


,auction_price
count,2504.000000
mean,32.596554
std,40.214437
min,3.090000
25%,10.085000
50%,16.200000
75%,39.782500
max,271.970000


Loaded file: C:\Users\work\OneDrive\Documents\level2-research\notebooks\04_auction_feature_table.csv
Loaded columns: 81


,code,date,auction_price,auction_trade_volume,auction_trade_amount,prev_day_return,has_auction_execution,large_order_amount_share,large_order_imbalance,post920_large_order_imbalance
0,000713.SZ,2026-01-05,6.63,27900.0,184977.0,NaN,1,0.297778,-1.000000,-1.0
1,000713.SZ,2026-01-06,6.70,24200.0,162140.0,0.009063,1,0.404593,-1.000000,-1.0
2,000713.SZ,2026-01-07,6.73,24800.0,166904.0,0.010479,1,0.295135,-0.727842,1.0
3,000713.SZ,2026-01-08,6.74,36240.0,244257.6,-0.001481,1,0.367789,-1.000000,0.0
4,000713.SZ,2026-01-09,6.73,54700.0,368131.0,0.002967,1,0.305438,-1.000000,0.0
5,000713.SZ,2026-01-12,6.77,58600.0,396722.0,0.002959,1,0.279774,-1.000000,-1.0
6,000713.SZ,2026-01-13,6.83,143300.0,978739.0,0.008850,1,0.210196,-0.808953,-1.0
7,000713.SZ,2026-01-14,6.80,50500.0,343400.0,-0.005848,1,0.318621,-0.835717,-1.0
8,000713.SZ,2026-01-15,6.77,60800.0,411616.0,-0.007353,1,0.435075,-1.000000,-1.0
9,000713.SZ,2026-01-16,6.82,26300.0,179366.0,0.005926,1,0.373856,-0.850613,-1.0


In [21]:
# ============================================================
# 22.2 Select Economically Distinct Auction Features
# ============================================================

auction_feature_candidates = [
    # --------------------------------------------------------
    # mechanism 1: auction price overreaction
    # --------------------------------------------------------
    "auction_price",
    "auction_vwap",
    "auction_min_price",
    "auction_max_price",
    "auction_price_consistent",

    # --------------------------------------------------------
    # Directional order pressure
    # --------------------------------------------------------
    "submit_imbalance",
    "post920_submit_imbalance",
    "last_minute_submit_imbalance",
    "final_depth_imbalance",
    "net_pressure_to_depth",
    "pressure_intensity_to_depth",

    # --------------------------------------------------------
    # Auction updating and cancellation
    # --------------------------------------------------------
    "post920_volume_share",
    "imbalance_shift",
    "imbalance_reversal",
    "cancel_volume_ratio",
    "cancel_rate_by_order",

    # --------------------------------------------------------
    # Belief divergence
    # --------------------------------------------------------
    "buy_price_divergence",
    "sell_price_divergence",
    "buy_sell_divergence_gap",

    # --------------------------------------------------------
    # Order-size concentration proxies
    # 注意：只能解释为 institutional-like proxy
    # --------------------------------------------------------
    "largest_order_share",
    "order_size_hhi",
    "large_order_amount_share",
    "large_order_count_share",
    "large_order_imbalance",
    "small_order_imbalance",
    "post920_large_order_imbalance",
    "top5_order_amount_share",
    "order_amount_hhi",
    "max_submit_size",
    "avg_submit_size",

    # --------------------------------------------------------
    # Auction price discovery
    # --------------------------------------------------------
    "auction_price_range",
    "auction_price_volatility",
    "last_minute_price_change",
    "price_reversal_rate",
    "n_price_revisions",
    "convergence_speed",
    "price_path_smoothness",

    # --------------------------------------------------------
    # Actual auction execution
    # --------------------------------------------------------
    "auction_trade_count",
    "auction_trade_volume",
    "auction_trade_amount",
    "has_auction_execution",

    # --------------------------------------------------------
    # Historical controls from File 4
    # --------------------------------------------------------
    "prev_day_return",
    "prev_day_amount",
    "rolling_5d_vol",
    "rolling_5d_amount"
]

auction_features = [
    feature
    for feature in auction_feature_candidates
    if feature in auction_feature_df.columns
]

missing_auction_features = [
    feature
    for feature in auction_feature_candidates
    if feature not in auction_feature_df.columns
]

print("Available auction features:", len(auction_features))
print(auction_features)

print("\nMissing candidate features:")
print(missing_auction_features)

assert "auction_price" in auction_features

Available auction features: 45
['auction_price', 'auction_vwap', 'auction_min_price', 'auction_max_price', 'auction_price_consistent', 'submit_imbalance', 'post920_submit_imbalance', 'last_minute_submit_imbalance', 'final_depth_imbalance', 'net_pressure_to_depth', 'pressure_intensity_to_depth', 'post920_volume_share', 'imbalance_shift', 'imbalance_reversal', 'cancel_volume_ratio', 'cancel_rate_by_order', 'buy_price_divergence', 'sell_price_divergence', 'buy_sell_divergence_gap', 'largest_order_share', 'order_size_hhi', 'large_order_amount_share', 'large_order_count_share', 'large_order_imbalance', 'small_order_imbalance', 'post920_large_order_imbalance', 'top5_order_amount_share', 'order_amount_hhi', 'max_submit_size', 'avg_submit_size', 'auction_price_range', 'auction_price_volatility', 'last_minute_price_change', 'price_reversal_rate', 'n_price_revisions', 'convergence_speed', 'price_path_smoothness', 'auction_trade_count', 'auction_trade_volume', 'auction_trade_amount', 'has_auction

In [22]:
# ============================================================
# Merge File 4 Auction Features into the Strict Opening Sample
# ============================================================
opening_model_df["date"] = pd.to_datetime(opening_model_df["date"]).dt.normalize()
auction_feature_df["date"] = pd.to_datetime(auction_feature_df["date"]).dt.normalize()

opening_auction_df = opening_model_df.merge(
    auction_feature_df,
    on=["code", "date"],
    how="left",
    validate="one_to_one",
    indicator=True,
)
merge_coverage = opening_auction_df["_merge"].eq("both").mean()
print("Auction merge coverage:", f"{merge_coverage:.2%}")
assert merge_coverage > 0.90
opening_auction_df = opening_auction_df.drop(columns="_merge")


Auction merge coverage: 100.00%


In [23]:
# ============================================================
# Boss-Directed, Leakage-Safe Feature Engineering
# ============================================================
opening_auction_df = opening_auction_df.sort_values(["code", "date"]).copy()
opening_auction_df["auction_return"] = (
    opening_auction_df["auction_price"] / opening_auction_df["pre_close"] - 1
)
opening_auction_df["abs_auction_return"] = opening_auction_df["auction_return"].abs()
opening_auction_df["auction_return_squared"] = opening_auction_df["auction_return"] ** 2

for raw_col, prefix_name in [
    ("auction_trade_volume", "auction_volume"),
    ("auction_trade_amount", "auction_amount"),
]:
    prior_mean = opening_auction_df.groupby("code")[raw_col].transform(
        lambda s: s.shift(1).rolling(5, min_periods=3).mean()
    )
    prior_std = opening_auction_df.groupby("code")[raw_col].transform(
        lambda s: s.shift(1).rolling(5, min_periods=3).std()
    )
    opening_auction_df[f"{prefix_name}_ratio_5d"] = safe_divide(
        opening_auction_df[raw_col], prior_mean
    )
    opening_auction_df[f"{prefix_name}_zscore_5d"] = safe_divide(
        opening_auction_df[raw_col] - prior_mean, prior_std
    )

for feature in [
    "auction_volume_ratio_5d", "final_depth_imbalance",
    "post920_submit_imbalance", "large_order_imbalance",
    "post920_large_order_imbalance",
]:
    if feature in opening_auction_df.columns:
        opening_auction_df[f"auction_return_x_{feature}"] = (
            opening_auction_df["auction_return"] * opening_auction_df[feature]
        )


In [24]:
# ============================================================
# Construct Strict 09:30 First-Quote Anchor
# Must run after qtick has been loaded
# ============================================================

assert "qtick" in globals(), (
    "qtick is not defined. Run the qtick loading cells first."
)

strict_open_ticks = qtick.copy()

# ------------------------------------------------------------
# Create datetime if necessary
# ------------------------------------------------------------

if "datetime" not in strict_open_ticks.columns:

    date_string = (
        pd.to_datetime(
            strict_open_ticks["date"],
            errors="coerce"
        )
        .dt.strftime("%Y-%m-%d")
    )

    time_string = (
        strict_open_ticks["time"]
        .astype(str)
        .str.replace("1970-01-01 ", "", regex=False)
        .str.replace("1900-01-01 ", "", regex=False)
    )

    strict_open_ticks["datetime"] = pd.to_datetime(
        date_string + " " + time_string,
        errors="coerce"
    )

# ------------------------------------------------------------
# Normalize key columns
# ------------------------------------------------------------

strict_open_ticks["date"] = (
    pd.to_datetime(
        strict_open_ticks["date"],
        errors="coerce"
    )
    .dt.normalize()
)

strict_open_ticks["code"] = (
    strict_open_ticks["code"]
    .astype(str)
    .str.strip()
    .str.upper()
)

# ------------------------------------------------------------
# Build valid midpoint
# ------------------------------------------------------------

strict_open_ticks["bp0"] = (
    pd.to_numeric(
        strict_open_ticks["bp0"],
        errors="coerce"
    )
    .replace(0, np.nan)
)

strict_open_ticks["ap0"] = (
    pd.to_numeric(
        strict_open_ticks["ap0"],
        errors="coerce"
    )
    .replace(0, np.nan)
)

strict_open_ticks["strict_open_mid_price"] = (
    strict_open_ticks["bp0"]
    + strict_open_ticks["ap0"]
) / 2

# ------------------------------------------------------------
# Select 09:30:00–09:30:59
# ------------------------------------------------------------

open_start_timestamp = (
    strict_open_ticks["date"]
    + pd.Timedelta(hours=9, minutes=30)
)

open_end_timestamp = (
    strict_open_ticks["date"]
    + pd.Timedelta(hours=9, minutes=31)
)

strict_open_ticks = strict_open_ticks[
    strict_open_ticks["datetime"].ge(open_start_timestamp)
    & strict_open_ticks["datetime"].lt(open_end_timestamp)
    & strict_open_ticks["strict_open_mid_price"].notna()
    & strict_open_ticks["strict_open_mid_price"].gt(0)
].copy()

# ------------------------------------------------------------
# First valid quote for every stock-date
# ------------------------------------------------------------

strict_open_anchor = (
    strict_open_ticks
    .sort_values(["code", "date", "datetime"])
    .groupby(["code", "date"], as_index=False)
    .first()[
        [
            "code",
            "date",
            "datetime",
            "strict_open_mid_price"
        ]
    ]
    .rename(
        columns={
            "datetime": "strict_open_timestamp"
        }
    )
)

strict_open_anchor["seconds_after_0930"] = (
    strict_open_anchor["strict_open_timestamp"]
    - (
        strict_open_anchor["date"]
        + pd.Timedelta(hours=9, minutes=30)
    )
).dt.total_seconds()

print("strict_open_anchor created:", "strict_open_anchor" in globals())
print("Shape:", strict_open_anchor.shape)
print("Stocks:", strict_open_anchor["code"].nunique())
print("Dates:", strict_open_anchor["date"].nunique())

display(strict_open_anchor.head(10))

display(
    strict_open_anchor["seconds_after_0930"]
    .describe(
        percentiles=[0.50, 0.90, 0.95, 0.99]
    )
)

strict_open_anchor created: True
Shape: (2514, 5)
Stocks: 40
Dates: 63


,code,date,strict_open_timestamp,strict_open_mid_price,seconds_after_0930
0,000713.SZ,2026-01-05,2026-01-05 09:30:00,6.635,0.0
1,000713.SZ,2026-01-06,2026-01-06 09:30:00,6.695,0.0
2,000713.SZ,2026-01-07,2026-01-07 09:30:00,6.745,0.0
3,000713.SZ,2026-01-08,2026-01-08 09:30:00,6.745,0.0
4,000713.SZ,2026-01-09,2026-01-09 09:30:00,6.735,0.0
5,000713.SZ,2026-01-12,2026-01-12 09:30:00,6.775,0.0
6,000713.SZ,2026-01-13,2026-01-13 09:30:00,6.855,0.0
7,000713.SZ,2026-01-14,2026-01-14 09:30:00,6.815,0.0
8,000713.SZ,2026-01-15,2026-01-15 09:30:00,6.765,0.0
9,000713.SZ,2026-01-16,2026-01-16 09:30:00,6.855,0.0


count    2514.000000
mean        0.798329
std         1.221370
min         0.000000
50%         0.000000
90%         3.000000
95%         3.000000
99%         5.000000
max         7.000000
Name: seconds_after_0930, dtype: float64

In [25]:
# ============================================================
# Merge Strict First Quote and Build Leakage-Safe Auction-to-Open Features
# ============================================================
auction_open_df = opening_auction_df.drop(
    columns=["strict_open_timestamp", "strict_open_mid_price", "seconds_after_0930"],
    errors="ignore",
).merge(
    strict_open_anchor,
    on=["code", "date"],
    how="left",
    validate="one_to_one",
)

MAX_OPEN_DELAY_SECONDS = 10
auction_open_df = auction_open_df.loc[
    auction_open_df["strict_open_mid_price"].notna()
    & auction_open_df["seconds_after_0930"].between(0, MAX_OPEN_DELAY_SECONDS)
].copy()
auction_open_df["actual_open_price"] = auction_open_df["strict_open_mid_price"]
auction_open_df["target_auction_to_open_return"] = (
    auction_open_df["actual_open_price"] / auction_open_df["auction_price"] - 1
)
print("Strict opening sample:", auction_open_df.shape)


Strict opening sample: (2506, 131)


# Part III. 严格 09:30→09:35 标签

主标签固定为：

```text
future_return_5m_raw = strict_mid_0935 / strict_mid_0930 - 1
```

两端价格均取目标时刻后允许范围内第一条有效双边报价的 midpoint。


In [56]:
# ============================================================
# Correct Strict 09:30 → 09:35 Mid-Price Return Label
# ============================================================

import numpy as np
import pandas as pd

strict_tick = qtick.copy()

# ------------------------------------------------------------
# 1. Basic timestamp and quote validation
# ------------------------------------------------------------

strict_tick["date"] = pd.to_datetime(
    strict_tick["date"]
).dt.normalize()

strict_tick["datetime"] = pd.to_datetime(
    strict_tick["datetime"]
)

for col in ["bp0", "ap0"]:
    strict_tick[col] = pd.to_numeric(
        strict_tick[col],
        errors="coerce"
    )

strict_tick = (
    strict_tick
    .replace([np.inf, -np.inf], np.nan)
    .dropna(
        subset=[
            "code",
            "date",
            "datetime",
            "bp0",
            "ap0"
        ]
    )
    .copy()
)

# Keep only valid two-sided quotes
strict_tick = strict_tick[
    (strict_tick["bp0"] > 0)
    &
    (strict_tick["ap0"] > 0)
    &
    (strict_tick["ap0"] >= strict_tick["bp0"])
].copy()

strict_tick["strict_mid_price"] = (
    strict_tick["bp0"]
    + strict_tick["ap0"]
) / 2

strict_tick = (
    strict_tick
    .sort_values(["code", "date", "datetime"])
    .drop_duplicates(
        subset=["code", "date", "datetime"],
        keep="last"
    )
)

# ------------------------------------------------------------
# 2. Construct target timestamps
# ------------------------------------------------------------

strict_tick["target_0930"] = (
    strict_tick["date"]
    + pd.Timedelta(hours=9, minutes=30)
)

strict_tick["target_0935"] = (
    strict_tick["date"]
    + pd.Timedelta(hours=9, minutes=35)
)

# Allow the first valid quote within 10 seconds
MAX_DELAY = pd.Timedelta(seconds=10)

# ------------------------------------------------------------
# 3. First valid quote at or after 09:30:00
# ------------------------------------------------------------

strict_0930 = strict_tick[
    (strict_tick["datetime"] >= strict_tick["target_0930"])
    &
    (
        strict_tick["datetime"]
        <= strict_tick["target_0930"] + MAX_DELAY
    )
].copy()

strict_0930 = (
    strict_0930
    .sort_values(["code", "date", "datetime"])
    .groupby(["code", "date"], as_index=False)
    .first()
)

strict_0930 = strict_0930[
    [
        "code",
        "date",
        "datetime",
        "bp0",
        "ap0",
        "strict_mid_price"
    ]
].rename(
    columns={
        "datetime": "strict_0930_timestamp",
        "bp0": "strict_0930_bp0",
        "ap0": "strict_0930_ap0",
        "strict_mid_price": "strict_mid_0930"
    }
)

# ------------------------------------------------------------
# 4. First valid quote at or after 09:35:00
# ------------------------------------------------------------

strict_0935 = strict_tick[
    (strict_tick["datetime"] >= strict_tick["target_0935"])
    &
    (
        strict_tick["datetime"]
        <= strict_tick["target_0935"] + MAX_DELAY
    )
].copy()

strict_0935 = (
    strict_0935
    .sort_values(["code", "date", "datetime"])
    .groupby(["code", "date"], as_index=False)
    .first()
)

strict_0935 = strict_0935[
    [
        "code",
        "date",
        "datetime",
        "bp0",
        "ap0",
        "strict_mid_price"
    ]
].rename(
    columns={
        "datetime": "strict_0935_timestamp",
        "bp0": "strict_0935_bp0",
        "ap0": "strict_0935_ap0",
        "strict_mid_price": "strict_mid_0935"
    }
)

# ------------------------------------------------------------
# 5. Merge strict start and future quotes
# ------------------------------------------------------------

strict_5m_label_df = strict_0930.merge(
    strict_0935,
    on=["code", "date"],
    how="inner",
    validate="one_to_one"
)

strict_5m_label_df[
    "strict_0930_delay_seconds"
] = (
    strict_5m_label_df["strict_0930_timestamp"]
    -
    (
        strict_5m_label_df["date"]
        + pd.Timedelta(hours=9, minutes=30)
    )
).dt.total_seconds()

strict_5m_label_df[
    "strict_0935_delay_seconds"
] = (
    strict_5m_label_df["strict_0935_timestamp"]
    -
    (
        strict_5m_label_df["date"]
        + pd.Timedelta(hours=9, minutes=35)
    )
).dt.total_seconds()

strict_5m_label_df["future_return_5m_strict"] = (
    strict_5m_label_df["strict_mid_0935"]
    / strict_5m_label_df["strict_mid_0930"]
    - 1
)

strict_5m_label_df["future_direction_5m_strict"] = np.select(
    [
        strict_5m_label_df[
            "future_return_5m_strict"
        ] > 0,

        strict_5m_label_df[
            "future_return_5m_strict"
        ] < 0
    ],
    [1, -1],
    default=0
)


# Official binary direction target. Zero returns are class 0.
strict_5m_label_df["future_direction_binary_5m"] = (
    strict_5m_label_df["future_return_5m_strict"] > 0
).astype(int)

print(
    "Strict 09:30–09:35 observations:",
    len(strict_5m_label_df)
)

print(
    "Stocks:",
    strict_5m_label_df["code"].nunique()
)

print(
    "Dates:",
    strict_5m_label_df["date"].nunique()
)

display(
    strict_5m_label_df[
        [
            "code",
            "date",
            "strict_0930_timestamp",
            "strict_mid_0930",
            "strict_0935_timestamp",
            "strict_mid_0935",
            "future_return_5m_strict",
            "future_direction_5m_strict"
        ]
    ].head(10)
)

Strict 09:30–09:35 observations: 2504
Stocks: 40
Dates: 63


,code,date,strict_0930_timestamp,strict_mid_0930,strict_0935_timestamp,strict_mid_0935,future_return_5m_strict,future_direction_5m_strict
0,000713.SZ,2026-01-05,2026-01-05 09:30:00,6.635,2026-01-05 09:35:00,6.625,-0.001507,-1
1,000713.SZ,2026-01-06,2026-01-06 09:30:00,6.695,2026-01-06 09:35:00,6.725,0.004481,1
2,000713.SZ,2026-01-07,2026-01-07 09:30:00,6.745,2026-01-07 09:35:00,6.735,-0.001483,-1
3,000713.SZ,2026-01-08,2026-01-08 09:30:00,6.745,2026-01-08 09:35:00,6.745,0.000000,0
4,000713.SZ,2026-01-09,2026-01-09 09:30:00,6.735,2026-01-09 09:35:00,6.745,0.001485,1
5,000713.SZ,2026-01-12,2026-01-12 09:30:00,6.775,2026-01-12 09:35:00,6.765,-0.001476,-1
6,000713.SZ,2026-01-13,2026-01-13 09:30:00,6.855,2026-01-13 09:35:00,6.885,0.004376,1
7,000713.SZ,2026-01-14,2026-01-14 09:30:00,6.815,2026-01-14 09:35:00,6.805,-0.001467,-1
8,000713.SZ,2026-01-15,2026-01-15 09:30:00,6.765,2026-01-15 09:35:00,6.765,0.000000,0
9,000713.SZ,2026-01-16,2026-01-16 09:30:00,6.855,2026-01-16 09:35:00,6.805,-0.007294,-1


In [57]:
# ============================================================
# Validate Strict Quote Timing
# ============================================================

timing_validation = (
    strict_5m_label_df[
        [
            "strict_0930_delay_seconds",
            "strict_0935_delay_seconds"
        ]
    ]
    .describe(
        percentiles=[
            0.50,
            0.90,
            0.95,
            0.99
        ]
    )
    .T
)

display(timing_validation)

assert (
    strict_5m_label_df[
        "strict_0930_delay_seconds"
    ].between(0, MAX_DELAY.total_seconds())
).all()

assert (
    strict_5m_label_df[
        "strict_0935_delay_seconds"
    ].between(0, MAX_DELAY.total_seconds())
).all()

,count,mean,std,min,50%,90%,95%,99%,max
strict_0930_delay_seconds,2504.0,0.798323,1.222893,0.0,0.0,3.0,3.0,5.0,7.0
strict_0935_delay_seconds,2504.0,0.555911,0.940960,0.0,0.0,2.0,2.0,4.0,10.0


In [58]:
# ============================================================
# Merge Corrected Strict Label into Auction Opening Dataset
# ============================================================

strict_label_cols = [
    "code",
    "date",
    "strict_0930_timestamp",
    "strict_mid_0930",
    "strict_0935_timestamp",
    "strict_mid_0935",
    "strict_0930_delay_seconds",
    "strict_0935_delay_seconds",
    "future_return_5m_strict",
    "future_direction_5m_strict"
]

auction_open_df_corrected = auction_open_df.merge(
    strict_5m_label_df[strict_label_cols],
    on=["code", "date"],
    how="inner",
    validate="one_to_one"
)

# Preserve old label explicitly
auction_open_df_corrected = (
    auction_open_df_corrected
    .rename(
        columns={
            "mid_price": "old_minute_end_mid_0930",
            "future_mid_5m": "old_minute_end_mid_0935",
            "future_return_5m": "old_minute_end_return_5m"
        }
    )
)

# Use the strict labels as the official labels
auction_open_df_corrected["mid_price"] = (
    auction_open_df_corrected["strict_mid_0930"]
)

auction_open_df_corrected["future_mid_5m"] = (
    auction_open_df_corrected["strict_mid_0935"]
)

auction_open_df_corrected["future_return_5m"] = (
    auction_open_df_corrected[
        "future_return_5m_strict"
    ]
)

auction_open_df_corrected["future_direction_5m"] = (
    auction_open_df_corrected[
        "future_direction_5m_strict"
    ]
)

print(
    "Original auction-opening observations:",
    len(auction_open_df)
)

print(
    "Corrected strict observations:",
    len(auction_open_df_corrected)
)

Original auction-opening observations: 2506
Corrected strict observations: 2504


In [60]:
auction_open_df = auction_open_df_corrected.copy()

In [62]:
# ============================================================
# FORCE RESET: Strict Label + Market-Adjusted Label
# ============================================================

# Ensure official raw label is the corrected strict return
auction_open_df["mid_price"] = (
    auction_open_df["strict_mid_0930"]
)

auction_open_df["future_mid_5m"] = (
    auction_open_df["strict_mid_0935"]
)

auction_open_df["future_return_5m"] = (
    auction_open_df["future_return_5m_strict"]
)

auction_open_df["future_direction_5m"] = (
    auction_open_df["future_direction_5m_strict"]
)

# Recalculate daily market return from corrected strict returns
auction_open_df["opening_market_return_5m"] = (
    auction_open_df
    .groupby("date")["future_return_5m"]
    .transform("mean")
)

# Recalculate excess return
auction_open_df[
    "future_opening_excess_return_5m"
] = (
    auction_open_df["future_return_5m"]
    -
    auction_open_df["opening_market_return_5m"]
)

# Recalculate daily rank
auction_open_df[
    "future_opening_excess_rank"
] = (
    auction_open_df
    .groupby("date")[
        "future_opening_excess_return_5m"
    ]
    .rank(
        pct=True,
        method="average"
    )
)

# ------------------------------------------------------------
# Hard validation
# ------------------------------------------------------------

auction_open_df["identity_check"] = (
    auction_open_df["future_return_5m"]
    -
    auction_open_df["opening_market_return_5m"]
    -
    auction_open_df[
        "future_opening_excess_return_5m"
    ]
)

assert (
    auction_open_df["identity_check"]
    .abs()
    .max()
    < 1e-12
)

assert (
    auction_open_df
    .groupby("date")[
        "future_opening_excess_return_5m"
    ]
    .mean()
    .abs()
    .max()
    < 1e-12
)

print("Corrected labels successfully rebuilt.")

Corrected labels successfully rebuilt.


## Strict-label data-quality audit

This audit consolidates sample size, missing values, duplicate keys, invalid
quotes, timestamp delays and extreme returns after the strict label has replaced
all earlier provisional targets.


In [63]:
STRICT_RETURN_COL = "future_return_5m"
STRICT_BINARY_COL = "future_direction_binary_5m"

auction_open_df[STRICT_BINARY_COL] = (
    auction_open_df[STRICT_RETURN_COL] > 0
).astype(int)

quality_columns = [
    "code", "date",
    "strict_0930_timestamp", "strict_mid_0930",
    "strict_0935_timestamp", "strict_mid_0935",
    STRICT_RETURN_COL, STRICT_BINARY_COL,
]
missing_quality_columns = [
    col for col in quality_columns if col not in auction_open_df.columns
]
assert not missing_quality_columns, (
    f"Strict data-quality audit missing columns: {missing_quality_columns}"
)

duplicate_code_dates = auction_open_df.duplicated(["code", "date"]).sum()
invalid_quote_rows = (
    auction_open_df["strict_mid_0930"].le(0)
    | auction_open_df["strict_mid_0935"].le(0)
).sum()
nonfinite_return_rows = (~np.isfinite(
    auction_open_df[STRICT_RETURN_COL]
)).sum()

strict_quality_summary = pd.DataFrame({
    "metric": [
        "observations", "stocks", "dates", "duplicate_code_date_rows",
        "invalid_mid_price_rows", "nonfinite_return_rows",
        "return_missing_ratio", "binary_positive_ratio",
        "zero_return_ratio", "abs_return_over_10pct_rows",
    ],
    "value": [
        len(auction_open_df),
        auction_open_df["code"].nunique(),
        auction_open_df["date"].nunique(),
        duplicate_code_dates,
        invalid_quote_rows,
        nonfinite_return_rows,
        auction_open_df[STRICT_RETURN_COL].isna().mean(),
        auction_open_df[STRICT_BINARY_COL].mean(),
        auction_open_df[STRICT_RETURN_COL].eq(0).mean(),
        auction_open_df[STRICT_RETURN_COL].abs().gt(0.10).sum(),
    ],
})

strict_missing_summary = (
    auction_open_df[quality_columns]
    .isna()
    .mean()
    .rename("missing_ratio")
    .sort_values(ascending=False)
    .to_frame()
)

display(strict_quality_summary)
display(strict_missing_summary)
assert duplicate_code_dates == 0
assert invalid_quote_rows == 0
assert nonfinite_return_rows == 0


,metric,value
0,observations,2504.000000
1,stocks,40.000000
2,dates,63.000000
3,duplicate_code_date_rows,0.000000
4,invalid_mid_price_rows,0.000000
5,nonfinite_return_rows,0.000000
6,return_missing_ratio,0.000000
7,binary_positive_ratio,0.481230
8,zero_return_ratio,0.030751
9,abs_return_over_10pct_rows,1.000000


,missing_ratio
code,0.0
date,0.0
strict_0930_timestamp,0.0
strict_mid_0930,0.0
strict_0935_timestamp,0.0
strict_mid_0935,0.0
future_return_5m,0.0
future_direction_binary_5m,0.0


In [ ]:
# ============================================================
# Build the clean factor-research base table
# ============================================================

factor_research_df = auction_open_df.copy()
factor_research_df["date"] = pd.to_datetime(
    factor_research_df["date"]
).dt.normalize()

factor_research_df["future_return_5m_raw"] = (
    pd.to_numeric(
        factor_research_df["strict_mid_0935"],
        errors="coerce",
    )
    /
    pd.to_numeric(
        factor_research_df["strict_mid_0930"],
        errors="coerce",
    )
    - 1
)

factor_research_df["future_direction_5m_raw"] = (
    factor_research_df["future_return_5m_raw"] > 0
).astype(int)

assert not factor_research_df.duplicated(["date", "code"]).any()
assert np.isfinite(
    factor_research_df["future_return_5m_raw"]
).all()

# Compatibility name for the already-tested strict-snapshot feature cells.
tail_df = factor_research_df.copy()

print("Factor-research observations:", len(factor_research_df))
print("Dates:", factor_research_df["date"].nunique())
print("Stocks:", factor_research_df["code"].nunique())


# Part IV. 预测时点可得因子与泄漏检查

只允许使用 09:30 前或严格 09:30 时点已经可见的信息。这里复用并整理
原 notebook 的严格盘口、竞价路径、相对市场/板块缺口和参与者结构特征。


In [89]:
# ============================================================
# Stage 2. Build Strict 09:30 Feature Registry
# ============================================================

# ------------------------------------------------------------
# 1. Features observable before or at strict 09:30
# ------------------------------------------------------------

candidate_feature_blocks = {

    # Previous-day and historical context
    "historical_state": [
        "prev_day_return",
        "prev_day_amount",
        "rolling_5d_return",
        "rolling_20d_return",
        "rolling_5d_vol",
        "rolling_20d_vol",
        "rolling_5d_amount",
        "rolling_20d_amount"
    ],

    # Auction price displacement
    "price_gap": [
        "auction_return",
        "abs_auction_return",
        "auction_return_squared",
        "positive_auction_gap",
        "negative_auction_gap",
        "volatility_adjusted_auction_gap"
    ],

    # Auction price-discovery path
    "price_discovery_path": [
        "auction_price_volatility",
        "n_price_revisions",
        "price_reversal_count",
        "avg_revision_magnitude",
        "median_revision_magnitude",
        "max_revision_magnitude",
        "convergence_speed",
        "path_smoothness",
        "auction_price_range",
        "last_minute_price_change"
    ],

    # Post-09:20 committed order flow
    "committed_order_flow": [
        "submit_imbalance",
        "post920_submit_imbalance",
        "last_minute_submit_imbalance",
        "imbalance_shift",
        "post920_volume_share",
        "last_minute_submit_volume_share",
        "last_minute_submit_count_share"
    ],

    # Order-size structure
    "order_size_structure": [
        "small_order_imbalance",
        "medium_order_imbalance",
        "large_order_imbalance",
        "very_large_order_imbalance",
        "post920_small_order_imbalance",
        "post920_medium_order_imbalance",
        "post920_large_order_imbalance",
        "post920_very_large_order_imbalance",
        "large_order_volume_share",
        "post920_large_order_share",
        "largest_order_share",
        "order_size_hhi"
    ],

    # Auction participation relative to history
    "relative_participation": [
        "auction_trade_count",
        "auction_trade_volume",
        "auction_trade_amount",
        "auction_volume_ratio_5d",
        "auction_volume_ratio_20d",
        "auction_amount_ratio_5d",
        "auction_amount_ratio_20d",
        "auction_volume_zscore_20d",
        "auction_amount_zscore_20d"
    ],

    # Final auction liquidity
    "final_auction_depth": [
        "final_bid_depth",
        "final_ask_depth",
        "final_total_depth",
        "final_depth_imbalance",
        "n_price_levels",
        "net_order_flow_to_depth",
        "large_order_flow_to_depth"
    ],

    # Case-driven gap-flow mechanisms
    "gap_flow_interactions": [
        "auction_gap_direction",
        "late_large_order_support",
        "large_order_imbalance_shift",
        "gap_x_post920_large_order",
        "gap_flow_alignment",
        "late_large_vs_small_divergence",
        "gap_opposition_strength",
        "volume_x_post920_large_order",
        "low_gap_indicator",
        "low_gap_x_late_large_buy",
        "low_gap_x_volume_supported_buy",
        "low_gap_x_opening_bid_support"
    ],

    # Auction result to strict opening transition
    "auction_to_open_transition": [
        "target_auction_to_open_return",
        "auction_to_open_return",
        "abs_auction_to_open_return",
        "auction_open_direction_agreement",
        "auction_open_reversal",
        "gap_absorption_ratio"
    ],

    # Strict first valid 09:30 snapshot
    "strict_opening_snapshot": [
        "relative_spread",
        "log_top_depth",
        "depth_imbalance",
        "microprice_deviation",
        "opening_relative_spread",
        "opening_log_top_depth",
        "opening_depth_imbalance",
        "opening_microprice_deviation"
    ]
}

# Board is categorical context, handled separately
categorical_context_features = [
    "board"
]

# ------------------------------------------------------------
# 2. Check availability
# ------------------------------------------------------------

registry_rows = []

for block_name, feature_list in (
    candidate_feature_blocks.items()
):

    for feature in feature_list:

        is_available = (
            feature in tail_df.columns
        )

        registry_rows.append({
            "feature_block": block_name,
            "feature": feature,
            "available": is_available,

            "n_non_missing": (
                tail_df[feature].notna().sum()
                if is_available
                else 0
            ),

            "coverage_ratio": (
                tail_df[feature].notna().mean()
                if is_available
                else 0.0
            ),

            "n_unique": (
                tail_df[feature].nunique(
                    dropna=True
                )
                if is_available
                else 0
            )
        })

feature_registry = pd.DataFrame(
    registry_rows
)

available_feature_registry = (
    feature_registry[
        feature_registry["available"]
    ]
    .sort_values(
        [
            "feature_block",
            "coverage_ratio",
            "feature"
        ],
        ascending=[
            True,
            False,
            True
        ]
    )
    .reset_index(drop=True)
)

missing_feature_registry = (
    feature_registry[
        ~feature_registry["available"]
    ]
    .sort_values(
        ["feature_block", "feature"]
    )
    .reset_index(drop=True)
)

print("Available candidate features:")
display(available_feature_registry)

print("Missing candidate features:")
display(missing_feature_registry)

Available candidate features:


,feature_block,feature,available,n_non_missing,coverage_ratio,n_unique
0,auction_to_open_transition,target_auction_to_open_return,True,2488,0.993610,2356
1,committed_order_flow,imbalance_shift,True,2504,1.000000,2504
2,committed_order_flow,last_minute_submit_count_share,True,2504,1.000000,2461
3,committed_order_flow,last_minute_submit_imbalance,True,2504,1.000000,2496
4,committed_order_flow,last_minute_submit_volume_share,True,2504,1.000000,2504
5,committed_order_flow,post920_submit_imbalance,True,2504,1.000000,2504
6,committed_order_flow,post920_volume_share,True,2504,1.000000,2504
7,committed_order_flow,submit_imbalance,True,2504,1.000000,2503
8,final_auction_depth,final_depth_imbalance,True,2504,1.000000,2410
9,final_auction_depth,final_total_depth,True,2504,1.000000,2049


Missing candidate features:


,feature_block,feature,available,n_non_missing,coverage_ratio,n_unique
0,auction_to_open_transition,abs_auction_to_open_return,False,0,0.0,0
1,auction_to_open_transition,auction_open_direction_agreement,False,0,0.0,0
2,auction_to_open_transition,auction_open_reversal,False,0,0.0,0
3,auction_to_open_transition,auction_to_open_return,False,0,0.0,0
4,auction_to_open_transition,gap_absorption_ratio,False,0,0.0,0
5,final_auction_depth,final_ask_depth,False,0,0.0,0
6,final_auction_depth,final_bid_depth,False,0,0.0,0
7,final_auction_depth,large_order_flow_to_depth,False,0,0.0,0
8,final_auction_depth,net_order_flow_to_depth,False,0,0.0,0
9,gap_flow_interactions,low_gap_indicator,False,0,0.0,0


In [90]:
# ============================================================
# Stage 2A1. Rebuild Strict 09:30 Snapshot Features
# ============================================================

strict_snapshot_source = qtick.copy()

strict_snapshot_source["date"] = pd.to_datetime(
    strict_snapshot_source["date"]
).dt.normalize()

strict_snapshot_source["datetime"] = pd.to_datetime(
    strict_snapshot_source["datetime"]
)

snapshot_numeric_cols = [
    "bp0",
    "ap0",
    "bv0",
    "av0",
    "bv1",
    "av1",
    "bv2",
    "av2",
    "bv3",
    "av3",
    "bv4",
    "av4"
]

for col in snapshot_numeric_cols:

    if col in strict_snapshot_source.columns:

        strict_snapshot_source[col] = pd.to_numeric(
            strict_snapshot_source[col],
            errors="coerce"
        )

# Valid best bid and ask are mandatory
strict_snapshot_source = (
    strict_snapshot_source
    .replace([np.inf, -np.inf], np.nan)
    .dropna(
        subset=[
            "code",
            "date",
            "datetime",
            "bp0",
            "ap0",
            "bv0",
            "av0"
        ]
    )
    .copy()
)

strict_snapshot_source = strict_snapshot_source[
    (strict_snapshot_source["bp0"] > 0)
    &
    (strict_snapshot_source["ap0"] > 0)
    &
    (
        strict_snapshot_source["ap0"]
        >= strict_snapshot_source["bp0"]
    )
    &
    (strict_snapshot_source["bv0"] >= 0)
    &
    (strict_snapshot_source["av0"] >= 0)
].copy()

# ------------------------------------------------------------
# 1. Strict target timestamp
# ------------------------------------------------------------

strict_snapshot_source["target_0930"] = (
    strict_snapshot_source["date"]
    + pd.Timedelta(
        hours=9,
        minutes=30
    )
)

MAX_OPENING_DELAY = pd.Timedelta(
    seconds=10
)

strict_snapshot_source = strict_snapshot_source[
    (
        strict_snapshot_source["datetime"]
        >= strict_snapshot_source["target_0930"]
    )
    &
    (
        strict_snapshot_source["datetime"]
        <=
        strict_snapshot_source["target_0930"]
        + MAX_OPENING_DELAY
    )
].copy()

# Preserve a coherent snapshot row:
# sort first, then keep the first complete row
strict_snapshot_df = (
    strict_snapshot_source
    .sort_values(
        ["code", "date", "datetime"]
    )
    .drop_duplicates(
        subset=["code", "date"],
        keep="first"
    )
    .copy()
)

strict_snapshot_df[
    "strict_opening_delay_seconds"
] = (
    strict_snapshot_df["datetime"]
    -
    strict_snapshot_df["target_0930"]
).dt.total_seconds()

# ------------------------------------------------------------
# 2. Strict opening midpoint and spread
# ------------------------------------------------------------

strict_snapshot_df["strict_opening_mid"] = (
    strict_snapshot_df["bp0"]
    + strict_snapshot_df["ap0"]
) / 2

strict_snapshot_df["strict_opening_spread"] = (
    strict_snapshot_df["ap0"]
    - strict_snapshot_df["bp0"]
)

strict_snapshot_df[
    "strict_opening_relative_spread"
] = (
    strict_snapshot_df["strict_opening_spread"]
    /
    strict_snapshot_df[
        "strict_opening_mid"
    ].replace(0, np.nan)
)

# ------------------------------------------------------------
# 3. Strict top-level depth
# ------------------------------------------------------------

strict_snapshot_df[
    "strict_opening_top_depth"
] = (
    strict_snapshot_df["bv0"]
    + strict_snapshot_df["av0"]
)

strict_snapshot_df[
    "strict_opening_log_top_depth"
] = np.log1p(
    strict_snapshot_df[
        "strict_opening_top_depth"
    ]
)

strict_snapshot_df[
    "strict_opening_depth_imbalance"
] = (
    strict_snapshot_df["bv0"]
    - strict_snapshot_df["av0"]
) / (
    strict_snapshot_df[
        "strict_opening_top_depth"
    ].replace(0, np.nan)
)

# ------------------------------------------------------------
# 4. Strict opening microprice
# ------------------------------------------------------------

strict_snapshot_df[
    "strict_opening_microprice"
] = (
    (
        strict_snapshot_df["ap0"]
        * strict_snapshot_df["bv0"]
    )
    +
    (
        strict_snapshot_df["bp0"]
        * strict_snapshot_df["av0"]
    )
) / (
    strict_snapshot_df[
        "strict_opening_top_depth"
    ].replace(0, np.nan)
)

strict_snapshot_df[
    "strict_opening_microprice_deviation"
] = (
    strict_snapshot_df[
        "strict_opening_microprice"
    ]
    /
    strict_snapshot_df[
        "strict_opening_mid"
    ].replace(0, np.nan)
    - 1
)

# ------------------------------------------------------------
# 5. Strict top-5 depth
# ------------------------------------------------------------

available_bid_depth_cols = [
    col for col in [
        "bv0", "bv1", "bv2", "bv3", "bv4"
    ]
    if col in strict_snapshot_df.columns
]

available_ask_depth_cols = [
    col for col in [
        "av0", "av1", "av2", "av3", "av4"
    ]
    if col in strict_snapshot_df.columns
]

strict_snapshot_df[
    "strict_opening_bid_depth_5"
] = (
    strict_snapshot_df[
        available_bid_depth_cols
    ]
    .sum(axis=1, min_count=1)
)

strict_snapshot_df[
    "strict_opening_ask_depth_5"
] = (
    strict_snapshot_df[
        available_ask_depth_cols
    ]
    .sum(axis=1, min_count=1)
)

strict_snapshot_df[
    "strict_opening_total_depth_5"
] = (
    strict_snapshot_df[
        "strict_opening_bid_depth_5"
    ]
    +
    strict_snapshot_df[
        "strict_opening_ask_depth_5"
    ]
)

strict_snapshot_df[
    "strict_opening_depth_imbalance_5"
] = (
    strict_snapshot_df[
        "strict_opening_bid_depth_5"
    ]
    -
    strict_snapshot_df[
        "strict_opening_ask_depth_5"
    ]
) / (
    strict_snapshot_df[
        "strict_opening_total_depth_5"
    ].replace(0, np.nan)
)

strict_snapshot_df[
    "strict_opening_log_total_depth_5"
] = np.log1p(
    strict_snapshot_df[
        "strict_opening_total_depth_5"
    ]
)

print(
    "Strict opening snapshots:",
    len(strict_snapshot_df)
)

print(
    "Maximum opening delay:",
    strict_snapshot_df[
        "strict_opening_delay_seconds"
    ].max()
)

Strict opening snapshots: 2514
Maximum opening delay: 7.0


In [91]:
# ============================================================
# Stage 2A2. Merge Strict Opening Features into Tail Dataset
# ============================================================

strict_opening_feature_cols = [
    "code",
    "date",
    "datetime",
    "strict_opening_delay_seconds",
    "strict_opening_mid",
    "strict_opening_spread",
    "strict_opening_relative_spread",
    "strict_opening_top_depth",
    "strict_opening_log_top_depth",
    "strict_opening_depth_imbalance",
    "strict_opening_microprice",
    "strict_opening_microprice_deviation",
    "strict_opening_bid_depth_5",
    "strict_opening_ask_depth_5",
    "strict_opening_total_depth_5",
    "strict_opening_depth_imbalance_5",
    "strict_opening_log_total_depth_5"
]

strict_snapshot_merge = (
    strict_snapshot_df[
        strict_opening_feature_cols
    ]
    .rename(
        columns={
            "datetime":
                "strict_opening_feature_timestamp"
        }
    )
)

# Remove old strict feature columns before merging,
# so rerunning this cell remains safe
existing_strict_feature_cols = [
    col
    for col in strict_snapshot_merge.columns
    if (
        col not in ["code", "date"]
        and col in tail_df.columns
    )
]

if existing_strict_feature_cols:

    tail_df = tail_df.drop(
        columns=existing_strict_feature_cols
    )

tail_df = tail_df.merge(
    strict_snapshot_merge,
    on=["code", "date"],
    how="inner",
    validate="one_to_one"
)

print(
    "Tail observations after strict snapshot merge:",
    len(tail_df)
)

Tail observations after strict snapshot merge: 2504


In [92]:
# ============================================================
# Stage 2A3. Rebuild Auction-to-Open Transition Features
# ============================================================

tail_df["auction_price"] = pd.to_numeric(
    tail_df["auction_price"],
    errors="coerce"
)

tail_df["auction_to_open_return"] = (
    tail_df["strict_opening_mid"]
    /
    tail_df["auction_price"].replace(0, np.nan)
    - 1
)

tail_df["abs_auction_to_open_return"] = (
    tail_df["auction_to_open_return"]
    .abs()
)

tail_df[
    "auction_open_direction_agreement"
] = (
    np.sign(tail_df["auction_return"])
    *
    np.sign(
        tail_df["auction_to_open_return"]
    )
)

# True when auction gap and auction-to-open move
# point in opposite directions
tail_df["auction_open_reversal"] = (
    tail_df[
        "auction_open_direction_agreement"
    ] < 0
).astype(int)

# Gap absorption:
# positive value means part of auction gap was corrected
tail_df["gap_absorption_ratio"] = np.where(
    tail_df["auction_return"].abs()
    > 1e-8,

    -tail_df["auction_to_open_return"]
    / tail_df["auction_return"],

    np.nan
)

# Avoid unstable ratios when auction_return is near zero
tail_df["gap_absorption_ratio"] = (
    tail_df["gap_absorption_ratio"]
    .clip(-5, 5)
)

In [93]:
# ============================================================
# Stage 2A4. Strict Opening Alignment Validation
# ============================================================

strict_alignment_check = pd.DataFrame({
    "metric": [
        "n_observations",
        "max_timestamp_delay_seconds",
        "mean_abs_mid_difference",
        "max_abs_mid_difference",
        "relative_spread_coverage",
        "depth_imbalance_coverage",
        "microprice_coverage"
    ],

    "value": [
        len(tail_df),

        tail_df[
            "strict_opening_delay_seconds"
        ].max(),

        (
            tail_df["strict_opening_mid"]
            - tail_df["strict_mid_0930"]
        ).abs().mean(),

        (
            tail_df["strict_opening_mid"]
            - tail_df["strict_mid_0930"]
        ).abs().max(),

        tail_df[
            "strict_opening_relative_spread"
        ].notna().mean(),

        tail_df[
            "strict_opening_depth_imbalance"
        ].notna().mean(),

        tail_df[
            "strict_opening_microprice_deviation"
        ].notna().mean()
    ]
})

display(strict_alignment_check)

assert (
    tail_df[
        "strict_opening_delay_seconds"
    ].max()
    <= 10
)

assert np.isclose(
    (
        tail_df["strict_opening_mid"]
        - tail_df["strict_mid_0930"]
    ).abs().max(),
    0.0,
    atol=1e-12
)

print(
    "Strict opening feature alignment passed."
)

,metric,value
0,n_observations,2504.0
1,max_timestamp_delay_seconds,7.0
2,mean_abs_mid_difference,0.0
3,max_abs_mid_difference,0.0
4,relative_spread_coverage,1.0
5,depth_imbalance_coverage,1.0
6,microprice_coverage,1.0


Strict opening feature alignment passed.


In [94]:
# ============================================================
# Update Feature Blocks after Strict Snapshot Correction
# ============================================================

candidate_feature_blocks[
    "auction_to_open_transition"
] = [
    "auction_to_open_return",
    "abs_auction_to_open_return",
    "auction_open_direction_agreement",
    "auction_open_reversal",
    "gap_absorption_ratio"
]

candidate_feature_blocks[
    "strict_opening_snapshot"
] = [
    "strict_opening_relative_spread",
    "strict_opening_log_top_depth",
    "strict_opening_depth_imbalance",
    "strict_opening_microprice_deviation",
    "strict_opening_log_total_depth_5",
    "strict_opening_depth_imbalance_5"
]

# Old minute-end snapshot features must not be used
deprecated_opening_features = [
    "relative_spread",
    "log_top_depth",
    "depth_imbalance",
    "microprice_deviation",
    "target_auction_to_open_return"
]

## Stage 2A3. Report-Informed Feature Engineering

这一节把投研报告中的机制转成 **strict 09:30 可观测 features**，但不直接照搬月频结论。

Feature design follows five rules:

1. **Residual gap**: distinguish market/board-wide opening moves from idiosyncratic auction displacement.
2. **Path location**: split total auction range into upper rejection and lower rejection.
3. **Path quality**: distinguish a stable price-discovery path from late jumps and repeated reversals.
4. **Participant structure**: separate large-order pressure from small-order noise and scale it by participation.
5. **No look-ahead**: every model feature must be observable by the strict 09:30 prediction timestamp.

The notebook deliberately does not fabricate multi-scale convergence variables from post-open qtick data. True 09:15–09:25 convergence features must be exported by File 4; if present, they are used automatically, otherwise they remain documented as unavailable candidates.


In [95]:
# ============================================================
# Report-Informed Strict 09:30 Feature Layer
# Safe to rerun: every engineered column is overwritten by name.
# ============================================================

import numpy as np
import pandas as pd


def _numeric_feature(data, column):
    """Return a numeric Series aligned to data.index; missing sources become NaN."""
    if column not in data.columns:
        return pd.Series(np.nan, index=data.index, dtype=float)
    return pd.to_numeric(data[column], errors="coerce")


def _safe_ratio(numerator, denominator):
    denominator = denominator.replace(0, np.nan)
    return (numerator / denominator).replace([np.inf, -np.inf], np.nan)


tail_df["date"] = pd.to_datetime(tail_df["date"]).dt.normalize()

# ------------------------------------------------------------
# A. Residual / idiosyncratic auction gap
# Source: residual-momentum logic from the momentum report.
# ------------------------------------------------------------
auction_gap = _numeric_feature(tail_df, "auction_return")

tail_df["market_median_auction_gap"] = (
    auction_gap.groupby(tail_df["date"]).transform("median")
)
tail_df["market_residual_auction_gap"] = (
    auction_gap - tail_df["market_median_auction_gap"]
)
tail_df["abs_market_residual_auction_gap"] = (
    tail_df["market_residual_auction_gap"].abs()
)
tail_df["auction_gap_daily_rank"] = (
    auction_gap.groupby(tail_df["date"]).rank(pct=True, method="average")
)

if "board" in tail_df.columns:
    tail_df["board_median_auction_gap"] = (
        auction_gap.groupby([tail_df["date"], tail_df["board"]]).transform("median")
    )
    tail_df["board_residual_auction_gap"] = (
        auction_gap - tail_df["board_median_auction_gap"]
    )
    tail_df["abs_board_residual_auction_gap"] = (
        tail_df["board_residual_auction_gap"].abs()
    )
    tail_df["auction_gap_board_rank"] = (
        auction_gap.groupby([tail_df["date"], tail_df["board"]])
        .rank(pct=True, method="average")
    )
else:
    for column in [
        "board_median_auction_gap",
        "board_residual_auction_gap",
        "abs_board_residual_auction_gap",
        "auction_gap_board_rank",
    ]:
        tail_df[column] = np.nan

# ------------------------------------------------------------
# B. Auction range location: rejection and final position
# Source: hidden structure of amplitude.
# ------------------------------------------------------------
auction_price = _numeric_feature(tail_df, "auction_price")
auction_min_price = _numeric_feature(tail_df, "auction_min_price")
auction_max_price = _numeric_feature(tail_df, "auction_max_price")
pre_close = _numeric_feature(tail_df, "pre_close")

valid_price_path = (
    auction_price.gt(0)
    & auction_min_price.gt(0)
    & auction_max_price.gt(0)
    & pre_close.gt(0)
    & auction_max_price.ge(auction_min_price)
    & auction_price.between(auction_min_price, auction_max_price, inclusive="both")
)

tail_df["auction_upper_rejection"] = np.where(
    valid_price_path,
    _safe_ratio(auction_max_price - auction_price, pre_close),
    np.nan,
)
tail_df["auction_lower_rejection"] = np.where(
    valid_price_path,
    _safe_ratio(auction_price - auction_min_price, pre_close),
    np.nan,
)
tail_df["auction_final_price_position"] = np.where(
    valid_price_path,
    _safe_ratio(
        auction_price - auction_min_price,
        auction_max_price - auction_min_price,
    ),
    np.nan,
)
tail_df["auction_rejection_asymmetry"] = (
    tail_df["auction_lower_rejection"]
    - tail_df["auction_upper_rejection"]
)

# ------------------------------------------------------------
# C. Path-quality proxies from existing File 4 aggregates
# ------------------------------------------------------------
auction_range = _numeric_feature(tail_df, "auction_price_range").abs()
last_minute_move = _numeric_feature(tail_df, "last_minute_price_change")
n_revisions = _numeric_feature(tail_df, "n_price_revisions")
reversal_count = _numeric_feature(tail_df, "price_reversal_count")

tail_df["auction_final_jump_share"] = _safe_ratio(
    last_minute_move.abs(),
    auction_range,
)
tail_df["auction_revision_reversal_ratio"] = _safe_ratio(
    reversal_count,
    n_revisions,
)
tail_df["auction_late_move_to_gap_ratio"] = _safe_ratio(
    last_minute_move.abs(),
    auction_gap.abs(),
)

# Keep extreme ratios numerically stable without using future labels.
for column in [
    "auction_final_jump_share",
    "auction_revision_reversal_ratio",
    "auction_late_move_to_gap_ratio",
]:
    tail_df[column] = tail_df[column].clip(lower=0, upper=10)

# ------------------------------------------------------------
# D. Large-order structure and gap-flow confirmation
# Source: large-trade microstructure and reversal mechanism.
# ------------------------------------------------------------
large_imbalance = _numeric_feature(tail_df, "large_order_imbalance")
small_imbalance = _numeric_feature(tail_df, "small_order_imbalance")
late_large_imbalance = _numeric_feature(tail_df, "post920_large_order_imbalance")
large_amount_share = _numeric_feature(tail_df, "large_order_amount_share")
large_volume_share = _numeric_feature(tail_df, "large_order_volume_share")
relative_volume = _numeric_feature(tail_df, "auction_volume_ratio_5d")
relative_amount = _numeric_feature(tail_df, "auction_amount_ratio_5d")

available_large_share = large_amount_share.where(
    large_amount_share.notna(),
    large_volume_share,
)

tail_df["large_vs_small_imbalance"] = (
    large_imbalance - small_imbalance
)
tail_df["late_large_vs_full_large_shift"] = (
    late_large_imbalance - large_imbalance
)
tail_df["large_flow_concentration"] = (
    late_large_imbalance.abs() * available_large_share
)

signed_gap_late_support = np.sign(auction_gap) * late_large_imbalance
tail_df["gap_large_flow_alignment_strength"] = (
    auction_gap.abs() * signed_gap_late_support.clip(lower=0)
)
tail_df["gap_large_flow_opposition_strength"] = (
    auction_gap.abs() * (-signed_gap_late_support).clip(lower=0)
)
tail_df["large_flow_x_relative_volume"] = (
    late_large_imbalance * relative_volume
)
tail_df["large_flow_x_relative_amount"] = (
    late_large_imbalance * relative_amount
)

# ------------------------------------------------------------
# E. APB-inspired terminal-price location (not the original APB)
# ------------------------------------------------------------
auction_vwap = _numeric_feature(tail_df, "auction_vwap")
tail_df["terminal_price_vs_auction_vwap"] = np.where(
    auction_price.gt(0) & auction_vwap.gt(0),
    np.log(auction_price / auction_vwap),
    np.nan,
)

# Mechanism interactions: direction comes from flow; instability from path.
tail_df["upper_rejection_x_large_sell"] = (
    tail_df["auction_upper_rejection"]
    * (-late_large_imbalance).clip(lower=0)
)
tail_df["lower_rejection_x_large_buy"] = (
    tail_df["auction_lower_rejection"]
    * late_large_imbalance.clip(lower=0)
)
tail_df["positive_gap_x_upper_rejection"] = (
    auction_gap.clip(lower=0)
    * tail_df["auction_upper_rejection"]
)
tail_df["negative_gap_x_lower_rejection"] = (
    (-auction_gap).clip(lower=0)
    * tail_df["auction_lower_rejection"]
)

# ------------------------------------------------------------
# F. Optional File 4 multi-scale features
# These are not computed from post-open qtick.
# ------------------------------------------------------------
optional_file4_features = [
    "auction_path_efficiency",
    "auction_trend_slope",
    "auction_trend_r2",
    "auction_multiscale_price_dispersion",
    "auction_multiscale_flow_dispersion",
    "auction_multiscale_flow_mean",
    "auction_price_flow_confirmation",
    "auction_executable_volume_bias",
]

report_engineered_features = [
    "market_residual_auction_gap",
    "abs_market_residual_auction_gap",
    "board_residual_auction_gap",
    "abs_board_residual_auction_gap",
    "auction_gap_daily_rank",
    "auction_gap_board_rank",
    "auction_upper_rejection",
    "auction_lower_rejection",
    "auction_final_price_position",
    "auction_rejection_asymmetry",
    "auction_final_jump_share",
    "auction_revision_reversal_ratio",
    "auction_late_move_to_gap_ratio",
    "large_vs_small_imbalance",
    "late_large_vs_full_large_shift",
    "large_flow_concentration",
    "gap_large_flow_alignment_strength",
    "gap_large_flow_opposition_strength",
    "large_flow_x_relative_volume",
    "large_flow_x_relative_amount",
    "terminal_price_vs_auction_vwap",
    "upper_rejection_x_large_sell",
    "lower_rejection_x_large_buy",
    "positive_gap_x_upper_rejection",
    "negative_gap_x_lower_rejection",
]

tail_df[report_engineered_features] = (
    tail_df[report_engineered_features]
    .replace([np.inf, -np.inf], np.nan)
)

print("Report-informed features created:", len(report_engineered_features))
print("Optional File 4 features already available:", [
    feature for feature in optional_file4_features
    if feature in tail_df.columns
])


Report-informed features created: 25
Optional File 4 features already available: []


In [96]:
# ============================================================
# Add Report-Informed Families to the Strict Feature Registry
# ============================================================

candidate_feature_blocks["residual_price_gap"] = [
    "market_residual_auction_gap",
    "abs_market_residual_auction_gap",
    "board_residual_auction_gap",
    "abs_board_residual_auction_gap",
    "auction_gap_daily_rank",
    "auction_gap_board_rank",
]

candidate_feature_blocks["path_location_and_rejection"] = [
    "auction_upper_rejection",
    "auction_lower_rejection",
    "auction_final_price_position",
    "auction_rejection_asymmetry",
    "terminal_price_vs_auction_vwap",
]

candidate_feature_blocks["path_quality_and_convergence"] = [
    "auction_final_jump_share",
    "auction_revision_reversal_ratio",
    "auction_late_move_to_gap_ratio",
    "auction_path_efficiency",
    "auction_trend_slope",
    "auction_trend_r2",
    "auction_multiscale_price_dispersion",
    "auction_multiscale_flow_dispersion",
    "auction_multiscale_flow_mean",
]

candidate_feature_blocks["large_order_microstructure"] = [
    "large_vs_small_imbalance",
    "late_large_vs_full_large_shift",
    "large_flow_concentration",
    "gap_large_flow_alignment_strength",
    "gap_large_flow_opposition_strength",
    "large_flow_x_relative_volume",
    "large_flow_x_relative_amount",
]

candidate_feature_blocks["report_mechanism_interactions"] = [
    "auction_price_flow_confirmation",
    "auction_executable_volume_bias",
    "upper_rejection_x_large_sell",
    "lower_rejection_x_large_buy",
    "positive_gap_x_upper_rejection",
    "negative_gap_x_lower_rejection",
]

print("Updated candidate feature blocks:", len(candidate_feature_blocks))


Updated candidate feature blocks: 15


In [97]:
# ============================================================
# Validate Report-Informed Features Before Screening
# ============================================================

report_feature_validation_rows = []

for feature in report_engineered_features + optional_file4_features:
    exists = feature in tail_df.columns
    series = (
        pd.to_numeric(tail_df[feature], errors="coerce")
        if exists else pd.Series(dtype=float)
    )
    report_feature_validation_rows.append({
        "feature": feature,
        "available": exists,
        "coverage_ratio": series.notna().mean() if exists else 0.0,
        "n_unique": series.nunique(dropna=True) if exists else 0,
        "has_infinite": bool(np.isinf(series.dropna()).any()) if exists else False,
    })

report_feature_validation = pd.DataFrame(report_feature_validation_rows)

if report_feature_validation["has_infinite"].any():
    bad_features = report_feature_validation.loc[
        report_feature_validation["has_infinite"], "feature"
    ].tolist()
    raise ValueError(f"Infinite engineered feature values detected: {bad_features}")

display(report_feature_validation)


,feature,available,coverage_ratio,n_unique,has_infinite
0,market_residual_auction_gap,True,0.993610,2299,False
1,abs_market_residual_auction_gap,True,0.993610,2257,False
2,board_residual_auction_gap,True,0.993610,2355,False
3,abs_board_residual_auction_gap,True,0.993610,2151,False
4,auction_gap_daily_rank,True,0.993610,201,False
5,auction_gap_board_rank,True,0.993610,47,False
6,auction_upper_rejection,True,0.938498,1,False
7,auction_lower_rejection,True,0.938498,1,False
8,auction_final_price_position,True,0.000000,0,False
9,auction_rejection_asymmetry,True,0.938498,1,False


In [98]:
# ============================================================
# Rebuild Feature Registry from Updated tail_df
# ============================================================

registry_rows = []

for block_name, feature_list in (
    candidate_feature_blocks.items()
):

    for feature in feature_list:

        is_available = (
            feature in tail_df.columns
        )

        registry_rows.append({
            "feature_block": block_name,
            "feature": feature,
            "available": is_available,

            "n_non_missing": (
                tail_df[feature].notna().sum()
                if is_available
                else 0
            ),

            "coverage_ratio": (
                tail_df[feature].notna().mean()
                if is_available
                else 0.0
            ),

            "n_unique": (
                tail_df[feature].nunique(
                    dropna=True
                )
                if is_available
                else 0
            )
        })

feature_registry = pd.DataFrame(
    registry_rows
)

available_feature_registry = (
    feature_registry[
        feature_registry["available"]
    ]
    .sort_values(
        [
            "feature_block",
            "coverage_ratio",
            "feature"
        ],
        ascending=[
            True,
            False,
            True
        ]
    )
    .reset_index(drop=True)
)

missing_feature_registry = (
    feature_registry[
        ~feature_registry["available"]
    ]
    .sort_values(
        ["feature_block", "feature"]
    )
    .reset_index(drop=True)
)

print(
    "Available updated features:",
    len(available_feature_registry)
)

display(available_feature_registry)

Available updated features: 79


,feature_block,feature,available,n_non_missing,coverage_ratio,n_unique
0,auction_to_open_transition,auction_open_reversal,True,2504,1.000000,2
1,auction_to_open_transition,abs_auction_to_open_return,True,2488,0.993610,2295
2,auction_to_open_transition,auction_open_direction_agreement,True,2488,0.993610,3
3,auction_to_open_transition,auction_to_open_return,True,2488,0.993610,2356
4,auction_to_open_transition,gap_absorption_ratio,True,2247,0.897364,2159
...,...,...,...,...,...,...
74,strict_opening_snapshot,strict_opening_depth_imbalance_5,True,2504,1.000000,2467
75,strict_opening_snapshot,strict_opening_log_top_depth,True,2504,1.000000,1233
76,strict_opening_snapshot,strict_opening_log_total_depth_5,True,2504,1.000000,2015
77,strict_opening_snapshot,strict_opening_microprice_deviation,True,2504,1.000000,2459


In [99]:
# ============================================================
# Stage 2A. Updated Block Availability Summary
# ============================================================

feature_block_summary = (
    feature_registry
    .groupby(
        "feature_block",
        observed=True
    )
    .agg(
        n_candidates=("feature", "size"),
        n_available=("available", "sum"),

        mean_coverage=(
            "coverage_ratio",
            "mean"
        )
    )
)

feature_block_summary[
    "availability_ratio"
] = (
    feature_block_summary["n_available"]
    /
    feature_block_summary["n_candidates"]
)

display(feature_block_summary)

,n_candidates,n_available,mean_coverage,availability_ratio
feature_block,,,,
auction_to_open_transition,5,5,0.975639,1.000000
committed_order_flow,7,7,1.000000,1.000000
final_auction_depth,7,3,0.428571,0.428571
gap_flow_interactions,12,8,0.660011,0.666667
historical_state,8,4,0.484125,0.500000
large_order_microstructure,7,7,0.984482,1.000000
order_size_structure,12,5,0.416667,0.416667
path_location_and_rejection,5,5,0.761821,1.000000
path_quality_and_convergence,9,3,0.311191,0.333333


In [100]:
# ============================================================
# Stage 2B. Build Updated Usable Feature Blocks
# ============================================================

MIN_FEATURE_COVERAGE = 0.80

usable_feature_blocks = {}

for block_name in candidate_feature_blocks:

    block_features = (
        available_feature_registry.loc[
            (
                available_feature_registry[
                    "feature_block"
                ] == block_name
            )
            &
            (
                available_feature_registry[
                    "coverage_ratio"
                ] >= MIN_FEATURE_COVERAGE
            )
            &
            (
                available_feature_registry[
                    "n_unique"
                ] >= 2
            ),
            "feature"
        ]
        .tolist()
    )

    # Additional safety:
    # remove deprecated minute-end features
    block_features = [
        feature
        for feature in block_features
        if feature not in deprecated_opening_features
    ]

    usable_feature_blocks[
        block_name
    ] = block_features

all_usable_features = []

for block_name, feature_list in (
    usable_feature_blocks.items()
):

    print(
        f"\n{block_name}: "
        f"{len(feature_list)} features"
    )

    for feature in feature_list:
        print("  -", feature)

    all_usable_features.extend(
        feature_list
    )

# Remove accidental duplicates while preserving order
all_usable_features = list(
    dict.fromkeys(all_usable_features)
)

print(
    "\nTotal usable continuous features:",
    len(all_usable_features)
)


historical_state: 4 features
  - prev_day_amount
  - prev_day_return
  - rolling_5d_amount
  - rolling_5d_vol

price_gap: 5 features
  - abs_auction_return
  - auction_return
  - auction_return_squared
  - negative_auction_gap
  - positive_auction_gap

price_discovery_path: 6 features
  - n_price_revisions
  - auction_price_range
  - price_reversal_count
  - auction_price_volatility
  - convergence_speed
  - last_minute_price_change

committed_order_flow: 7 features
  - imbalance_shift
  - last_minute_submit_count_share
  - last_minute_submit_imbalance
  - last_minute_submit_volume_share
  - post920_submit_imbalance
  - post920_volume_share
  - submit_imbalance

order_size_structure: 5 features
  - large_order_imbalance
  - largest_order_share
  - order_size_hhi
  - post920_large_order_imbalance
  - small_order_imbalance

relative_participation: 5 features
  - auction_trade_amount
  - auction_trade_count
  - auction_trade_volume
  - auction_amount_ratio_5d
  - auction_volume_ratio_5d


In [101]:
# ============================================================
# Stage 2C. Updated Leakage-Control Check
# ============================================================

forbidden_model_columns = [
    "strict_mid_0935",
    "future_mid_5m",
    "future_return_5m",
    "future_return_5m_strict",
    "future_direction_5m",
    "future_direction_5m_strict",

    "opening_market_return_5m",
    "future_opening_excess_return_5m",
    "future_opening_excess_rank",

    "future_return_5m_bps",
    "future_excess_return_5m_bps",

    "tail_class",
    "tail_class_label",
    "is_extreme_move",
    "is_large_positive",
    "is_large_negative",

    "old_minute_end_mid_0930",
    "old_minute_end_mid_0935",
    "old_minute_end_return_5m"
]

detected_leakage = [
    feature
    for feature in all_usable_features
    if feature in forbidden_model_columns
]

detected_deprecated_features = [
    feature
    for feature in all_usable_features
    if feature in deprecated_opening_features
]

assert not detected_leakage, (
    f"Future leakage detected: {detected_leakage}"
)

assert not detected_deprecated_features, (
    "Deprecated minute-end features detected: "
    f"{detected_deprecated_features}"
)

assert all(
    feature in tail_df.columns
    for feature in all_usable_features
)

print("Updated feature registry passed.")
print(
    "Usable continuous features:",
    len(all_usable_features)
)

print(
    "Categorical context features:",
    categorical_context_features
)

# Report-informed hard guard: no 09:30–09:35 realized information may enter predictors.
forbidden_post_open_predictors = {
    "return_1m", "return_3m", "return_5m",
    "volatility_3m", "volatility_5m",
    "volume_1m_x", "volume_3m_x", "volume_5m_x",
    "volume_acceleration_x", "turnover_1m", "turnover_3m",
    "turnover_5m", "trade_intensity_5m",
    "signed_trade_volume_5m", "hybrid_signed_volume_5m",
}

post_open_leakage_in_registry = sorted(
    set(all_usable_features) & forbidden_post_open_predictors
)

assert not post_open_leakage_in_registry, (
    "Post-open predictors detected in strict feature registry: "
    f"{post_open_leakage_in_registry}"
)

print("Strict 09:30 post-open leakage guard: PASSED")


Updated feature registry passed.
Usable continuous features: 71
Categorical context features: ['board']
Strict 09:30 post-open leakage guard: PASSED


In [ ]:
# Synchronize the clean research table after strict-snapshot engineering.
factor_research_df = tail_df.copy()
factor_research_df["future_return_5m_raw"] = (
    factor_research_df["strict_mid_0935"]
    / factor_research_df["strict_mid_0930"]
    - 1
)
factor_research_df["future_direction_5m_raw"] = (
    factor_research_df["future_return_5m_raw"] > 0
).astype(int)


# Part V. 基础因子统一排名

本阶段只做描述性筛选，不把全样本最高 IC 当作最终证据。二元机制因子
允许进入评价，因此最小唯一值数量为 2。


In [ ]:
# ============================================================
# 18. Define the Official Strict Raw-Return Target
# ============================================================

RAW_RETURN_TARGET = (
    "future_return_5m_strict"
)

RAW_DIRECTION_TARGET = (
    "future_direction_binary_5m"
)

required_raw_target_columns = [
    "code",
    "date",
    "strict_mid_0930",
    "strict_mid_0935",
    RAW_RETURN_TARGET,
]

missing_raw_target_columns = [
    col
    for col in required_raw_target_columns
    if col not in tail_df.columns
]

if missing_raw_target_columns:
    raise KeyError(
        "Raw-return research is missing columns: "
        f"{missing_raw_target_columns}"
    )

raw_model_df = factor_research_df.copy()

raw_model_df["date"] = (
    pd.to_datetime(
        raw_model_df["date"]
    )
    .dt.normalize()
)

raw_model_df["code"] = (
    raw_model_df["code"]
    .astype(str)
    .str.strip()
    .str.upper()
)

# Rebuild the official raw target directly from strict prices.
raw_model_df[
    "future_return_5m_raw"
] = (
    raw_model_df[
        "strict_mid_0935"
    ]
    / raw_model_df[
        "strict_mid_0930"
    ]
    - 1
)

raw_model_df[
    "future_direction_5m_raw"
] = (
    raw_model_df[
        "future_return_5m_raw"
    ] > 0
).astype(int)

# Validate against the existing strict target.
raw_target_identity_error = (
    raw_model_df[
        "future_return_5m_raw"
    ]
    - raw_model_df[
        RAW_RETURN_TARGET
    ]
).abs().max()

assert (
    raw_target_identity_error
    < 1e-12
), (
    "Rebuilt raw target does not match "
    "future_return_5m_strict."
)

RAW_MODEL_TARGET = (
    "future_return_5m_raw"
)

RAW_MODEL_DIRECTION_TARGET = (
    "future_direction_5m_raw"
)

print(
    "Official raw-return target:",
    RAW_MODEL_TARGET,
)

print(
    "Maximum strict-label identity error:",
    raw_target_identity_error,
)


In [ ]:
# ============================================================
# Stage 18A-0. Attach the Raw 09:30→09:35 Return Target
# ============================================================

RAW_RETURN_TARGET = "future_return_5m_raw"

# Reconstruct the target directly from strict prices if necessary
if RAW_RETURN_TARGET not in tail_df.columns:

    possible_price_pairs = [
        ("strict_mid_0930", "strict_mid_0935"),
        ("strict_open_mid", "strict_mid_0935"),
        ("price_0930_strict", "price_0935_strict"),
    ]

    selected_price_pair = next(
        (
            (price_t, price_t5)
            for price_t, price_t5 in possible_price_pairs
            if (
                price_t in tail_df.columns
                and price_t5 in tail_df.columns
            )
        ),
        None
    )

    if selected_price_pair is not None:

        price_t, price_t5 = selected_price_pair

        tail_df[RAW_RETURN_TARGET] = (
            pd.to_numeric(
                tail_df[price_t],
                errors="coerce"
            )
            /
            pd.to_numeric(
                tail_df[price_t5],
                errors="coerce"
            )
        )

        # Correct formula: price(t+5) / price(t) - 1
        tail_df[RAW_RETURN_TARGET] = (
            pd.to_numeric(
                tail_df[price_t5],
                errors="coerce"
            )
            /
            pd.to_numeric(
                tail_df[price_t],
                errors="coerce"
            )
            - 1
        )

        print(
            f"Constructed {RAW_RETURN_TARGET} from "
            f"{price_t5} / {price_t} - 1"
        )

    else:
        # Search for an already-created target DataFrame
        target_source = None

        for object_name, object_value in list(globals().items()):

            if (
                isinstance(object_value, pd.DataFrame)
                and RAW_RETURN_TARGET in object_value.columns
                and {"date", "code"}.issubset(object_value.columns)
            ):
                target_source = (
                    object_name,
                    object_value[
                        ["date", "code", RAW_RETURN_TARGET]
                    ].copy()
                )
                break

        if target_source is None:
            raise KeyError(
                f"{RAW_RETURN_TARGET} is not in tail_df, "
                "strict 09:30/09:35 price columns were not found, "
                "and no existing DataFrame contains the target."
            )

        source_name, target_labels = target_source

        target_labels["date"] = pd.to_datetime(
            target_labels["date"]
        )
        tail_df["date"] = pd.to_datetime(
            tail_df["date"]
        )

        target_labels = (
            target_labels
            .drop_duplicates(["date", "code"])
        )

        tail_df = tail_df.merge(
            target_labels,
            on=["date", "code"],
            how="left",
            validate="one_to_one"
        )

        print(
            f"Merged {RAW_RETURN_TARGET} into tail_df "
            f"from {source_name}"
        )

assert RAW_RETURN_TARGET in tail_df.columns

print(
    tail_df[RAW_RETURN_TARGET]
    .describe(
        percentiles=[0.01, 0.05, 0.50, 0.95, 0.99]
    )
)

print(
    "Target coverage:",
    tail_df[RAW_RETURN_TARGET].notna().mean()
)


In [ ]:
# ============================================================
# 18A. Leakage-Safe Raw-Return Feature Registry
# ============================================================

raw_return_feature_blocks = {

    # --------------------------------------------------------
    # Historical information available before current date
    # --------------------------------------------------------
    "historical_state": [
        "prev_day_return",
        "prev_day_amount",
        "rolling_5d_return",
        "rolling_20d_return",
        "rolling_5d_vol",
        "rolling_20d_vol",
        "rolling_5d_amount",
        "rolling_20d_amount",
    ],

    # --------------------------------------------------------
    # Auction price discovery
    # --------------------------------------------------------
    "auction_price_discovery": [
        "auction_return",
        "abs_auction_return",
        "auction_return_squared",
        "positive_auction_gap",
        "negative_auction_gap",
        "volatility_adjusted_auction_gap",
        "market_residual_auction_gap",
        "board_residual_auction_gap",
        "abs_market_residual_auction_gap",
        "abs_board_residual_auction_gap",
        "auction_price_range",
        "auction_final_jump_share",
        "auction_revision_reversal_ratio",
        "auction_late_move_to_gap_ratio",
    ],

    # --------------------------------------------------------
    # Auction participation intensity
    # --------------------------------------------------------
    "auction_participation": [
        "auction_trade_volume",
        "auction_trade_amount",
        "auction_trade_count",
        "auction_volume_ratio_5d",
        "auction_amount_ratio_5d",
        "auction_volume_zscore_5d",
        "auction_amount_zscore_5d",
        "post920_volume_share",
        "last_minute_submit_volume_share",
        "last_minute_submit_count_share",
    ],

    # --------------------------------------------------------
    # Order support and cancellation
    # --------------------------------------------------------
    "order_support": [
        "submit_imbalance",
        "post920_submit_imbalance",
        "last_minute_submit_imbalance",
        "final_depth_imbalance",
        "net_pressure_to_depth",
        "pressure_intensity_to_depth",
        "cancel_imbalance",
        "cancel_to_submit_ratio",
        "n_price_revisions",
    ],

    # --------------------------------------------------------
    # Large/small participant proxies
    # --------------------------------------------------------
    "participant_structure": [
        "large_order_imbalance",
        "small_order_imbalance",
        "post920_large_order_imbalance",
        "large_order_amount_share",
        "large_order_count_share",
        "large_flow_concentration",
        "large_vs_small_imbalance",
        "late_large_vs_full_large_shift",
        "gap_large_flow_alignment_strength",
        "gap_large_flow_opposition_strength",
    ],

    # --------------------------------------------------------
    # Strict 09:30 snapshot
    # --------------------------------------------------------
    "strict_opening_snapshot": [
        "strict_opening_relative_spread",
        "strict_opening_log_top_depth",
        "strict_opening_depth_imbalance",
        "strict_opening_microprice_deviation",
        "strict_opening_log_total_depth_5",
        "strict_opening_depth_imbalance_5",
    ],

    # --------------------------------------------------------
    # Auction-to-open transition
    # Observable at the strict 09:30 prediction timestamp
    # --------------------------------------------------------
    "auction_to_open_transition": [
        "auction_to_open_return",
        "abs_auction_to_open_return",
        "auction_open_direction_agreement",
        "auction_open_reversal",
        "gap_absorption_ratio",
    ],
}

# ------------------------------------------------------------
# Keep only columns that actually exist.
# ------------------------------------------------------------

available_raw_feature_blocks = {}

for block_name, feature_list in (
    raw_return_feature_blocks.items()
):

    available_features = [
        feature
        for feature in feature_list
        if feature in raw_model_df.columns
    ]

    available_raw_feature_blocks[
        block_name
    ] = available_features

raw_candidate_features = list(
    dict.fromkeys(
        feature
        for feature_list
        in available_raw_feature_blocks.values()
        for feature in feature_list
    )
)

# ------------------------------------------------------------
# Explicit leakage guards
# ------------------------------------------------------------

raw_forbidden_exact_columns = {
    "strict_mid_0935",
    "future_mid_5m",
    "future_return_5m",
    "future_return_5m_strict",
    "future_return_5m_raw",
    "future_direction_5m",
    "future_direction_5m_strict",
    "future_direction_5m_raw",
    "future_direction_binary_5m",
    "future_signed_return_5m",
    "future_opening_excess_return_5m",
    "future_opening_excess_rank",
    "opening_market_return_5m",
}

raw_forbidden_prefixes = (
    "future_",
    "actual_",
    "target_",
)

raw_leakage_candidates = [
    feature
    for feature in raw_candidate_features
    if (
        feature in raw_forbidden_exact_columns
        or feature.startswith(
            raw_forbidden_prefixes
        )
    )
]

if (
    "FORBIDDEN_POST_OPEN_FEATURES"
    in globals()
):

    raw_leakage_candidates.extend([
        feature
        for feature in raw_candidate_features
        if feature
        in FORBIDDEN_POST_OPEN_FEATURES
    ])

raw_leakage_candidates = sorted(
    set(raw_leakage_candidates)
)

assert not raw_leakage_candidates, (
    "Raw feature registry contains future "
    "or post-open information: "
    f"{raw_leakage_candidates}"
)

print(
    "Available raw-return feature blocks:"
)

for block_name, feature_list in (
    available_raw_feature_blocks.items()
):

    print(
        f"{block_name}: "
        f"{len(feature_list)} features"
    )

    print(
        feature_list
    )

print(
    "\nTotal raw candidate features:",
    len(raw_candidate_features),
)


In [ ]:
# ============================================================
# 18B. Construct the Raw-Return Master Modeling Table
# ============================================================

raw_context_candidates = [
    "board",
    "price_group",
]

available_raw_context_columns = [
    col
    for col in raw_context_candidates
    if col in raw_model_df.columns
]

raw_master_columns = list(
    dict.fromkeys(
        [
            "code",
            "date",
            "strict_mid_0930",
            "strict_mid_0935",
            RAW_MODEL_TARGET,
            RAW_MODEL_DIRECTION_TARGET,
        ]
        + raw_candidate_features
        + available_raw_context_columns
    )
)

raw_return_master_df = (
    raw_model_df[
        raw_master_columns
    ]
    .copy()
    .sort_values(
        [
            "date",
            "code",
        ]
    )
    .reset_index(drop=True)
)

# Convert model features to numeric.
for feature in raw_candidate_features:

    raw_return_master_df[
        feature
    ] = pd.to_numeric(
        raw_return_master_df[
            feature
        ],
        errors="coerce",
    )

raw_return_master_df = (
    raw_return_master_df
    .replace(
        [np.inf, -np.inf],
        np.nan,
    )
)

print(
    "Raw-return master shape:",
    raw_return_master_df.shape,
)

print(
    "Raw-return dates:",
    raw_return_master_df[
        "date"
    ].nunique(),
)

print(
    "Raw-return stocks:",
    raw_return_master_df[
        "code"
    ].nunique(),
)

display(
    raw_return_master_df.head()
)


## 19. Raw-Return Data Quality Audit

The following audit checks whether the strict raw-return sample is suitable for
rolling out-of-sample modeling.

The audit covers:

- duplicate stock-date observations;
- missing strict 09:30 and 09:35 prices;
- invalid prices;
- non-finite returns;
- extreme returns;
- feature coverage;
- feature uniqueness;
- stock and date counts.

Features are not removed solely because of weak full-sample IC. The feature
registry is based primarily on economic mechanism and information timing.
Full-sample IC is treated as a descriptive diagnostic rather than a leakage-safe
model-selection procedure.


In [ ]:
# ============================================================
# 19A. Raw-Return Data Quality Audit
# ============================================================

raw_duplicate_count = (
    raw_return_master_df
    .duplicated(
        subset=[
            "code",
            "date",
        ]
    )
    .sum()
)

raw_invalid_price_count = (
    raw_return_master_df[
        "strict_mid_0930"
    ].le(0)
    | raw_return_master_df[
        "strict_mid_0935"
    ].le(0)
).sum()

raw_nonfinite_target_count = (
    ~np.isfinite(
        raw_return_master_df[
            RAW_MODEL_TARGET
        ]
    )
).sum()

raw_extreme_return_count = (
    raw_return_master_df[
        RAW_MODEL_TARGET
    ]
    .abs()
    .gt(0.10)
    .sum()
)

raw_quality_summary = pd.DataFrame({
    "metric": [
        "n_observations",
        "n_dates",
        "n_stocks",
        "duplicate_code_date_rows",
        "invalid_strict_price_rows",
        "nonfinite_target_rows",
        "target_missing_ratio",
        "positive_return_ratio",
        "zero_return_ratio",
        "absolute_return_over_10pct_rows",
    ],
    "value": [
        len(
            raw_return_master_df
        ),

        raw_return_master_df[
            "date"
        ].nunique(),

        raw_return_master_df[
            "code"
        ].nunique(),

        raw_duplicate_count,

        raw_invalid_price_count,

        raw_nonfinite_target_count,

        raw_return_master_df[
            RAW_MODEL_TARGET
        ].isna().mean(),

        raw_return_master_df[
            RAW_MODEL_DIRECTION_TARGET
        ].mean(),

        raw_return_master_df[
            RAW_MODEL_TARGET
        ].eq(0).mean(),

        raw_extreme_return_count,
    ],
})

display(
    raw_quality_summary
)

assert raw_duplicate_count == 0

assert raw_invalid_price_count == 0

assert raw_nonfinite_target_count == 0


In [ ]:
# ============================================================
# 19B. Feature Coverage and Uniqueness
# ============================================================

raw_feature_quality_rows = []

for feature in raw_candidate_features:

    feature_series = (
        raw_return_master_df[
            feature
        ]
    )

    raw_feature_quality_rows.append({
        "feature":
            feature,

        "feature_block":
            next(
                (
                    block_name
                    for block_name, features
                    in available_raw_feature_blocks.items()
                    if feature in features
                ),
                "Unknown",
            ),

        "coverage_ratio":
            feature_series.notna().mean(),

        "missing_ratio":
            feature_series.isna().mean(),

        "n_unique":
            feature_series.nunique(
                dropna=True
            ),

        "mean":
            feature_series.mean(),

        "std":
            feature_series.std(),

        "minimum":
            feature_series.min(),

        "maximum":
            feature_series.max(),
    })

raw_feature_quality = (
    pd.DataFrame(
        raw_feature_quality_rows
    )
    .sort_values(
        [
            "coverage_ratio",
            "n_unique",
        ],
        ascending=[
            True,
            True,
        ],
    )
    .reset_index(drop=True)
)

display(
    raw_feature_quality
)

RAW_MIN_FEATURE_COVERAGE = 0.50
RAW_MIN_FEATURE_UNIQUE = 2

raw_model_features = (
    raw_feature_quality[
        raw_feature_quality[
            "coverage_ratio"
        ].ge(
            RAW_MIN_FEATURE_COVERAGE
        )
        & raw_feature_quality[
            "n_unique"
        ].ge(
            RAW_MIN_FEATURE_UNIQUE
        )
    ][
        "feature"
    ]
    .tolist()
)

raw_excluded_features = (
    raw_feature_quality[
        ~raw_feature_quality[
            "feature"
        ].isin(
            raw_model_features
        )
    ]
)

print(
    "Raw model features passing quality guard:",
    len(raw_model_features),
)

print(
    "Excluded low-coverage/low-variation features:",
    len(raw_excluded_features),
)

display(
    raw_excluded_features
)


## 20. Strict Raw Five-Minute Return Distribution

This section describes the official strict 09:30→09:35 raw-return target.

The distribution is evaluated before any model fitting. The target is not
winsorized or clipped in the official label table. Robust models may reduce the
influence of extreme observations during training, but evaluation always uses
the original unmodified return.


In [ ]:
# ============================================================
# 20A. Raw Five-Minute Return Distribution
# ============================================================

raw_target_summary = (
    raw_return_master_df[
        RAW_MODEL_TARGET
    ]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99,
        ]
    )
)

display(
    raw_target_summary
)

raw_direction_summary = (
    raw_return_master_df[
        RAW_MODEL_DIRECTION_TARGET
    ]
    .value_counts(
        normalize=False
    )
    .sort_index()
    .rename("count")
    .to_frame()
)

raw_direction_summary[
    "ratio"
] = (
    raw_direction_summary[
        "count"
    ]
    / raw_direction_summary[
        "count"
    ].sum()
)

raw_direction_summary.index = (
    raw_direction_summary.index.map({
        0: "Non-positive",
        1: "Positive",
    })
)

display(
    raw_direction_summary
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 5),
)

# Full distribution
axes[0].hist(
    raw_return_master_df[
        RAW_MODEL_TARGET
    ].dropna() * 10_000,
    bins=80,
    alpha=0.8,
)

axes[0].axvline(
    0,
    color="black",
    linestyle="--",
    linewidth=1,
)

axes[0].set_title(
    "Strict 09:30–09:35 Raw Return Distribution"
)

axes[0].set_xlabel(
    "Future raw return (bps)"
)

axes[0].set_ylabel(
    "Observations"
)

axes[0].grid(
    alpha=0.3
)

# Winsorized view for visualization only
raw_plot_return_bps = (
    raw_return_master_df[
        RAW_MODEL_TARGET
    ]
    .clip(
        lower=raw_return_master_df[
            RAW_MODEL_TARGET
        ].quantile(0.01),

        upper=raw_return_master_df[
            RAW_MODEL_TARGET
        ].quantile(0.99),
    )
    * 10_000
)

axes[1].hist(
    raw_plot_return_bps.dropna(),
    bins=60,
    alpha=0.8,
)

axes[1].axvline(
    0,
    color="black",
    linestyle="--",
    linewidth=1,
)

axes[1].set_title(
    "Raw Return Distribution: 1%–99% View"
)

axes[1].set_xlabel(
    "Future raw return (bps)"
)

axes[1].set_ylabel(
    "Observations"
)

axes[1].grid(
    alpha=0.3
)

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 20B. Feature Missing-Rate Visualization
# ============================================================

raw_missing_plot = (
    raw_feature_quality
    .sort_values(
        "missing_ratio",
        ascending=True,
    )
    .tail(25)
)

plt.figure(
    figsize=(10, 8)
)

plt.barh(
    raw_missing_plot[
        "feature"
    ],
    raw_missing_plot[
        "missing_ratio"
    ],
)

plt.xlabel(
    "Missing ratio"
)

plt.ylabel(
    "Feature"
)

plt.title(
    "Top Raw-Model Feature Missing Ratios"
)

plt.grid(
    axis="x",
    alpha=0.3,
)

plt.tight_layout()
plt.show()


## 21 描述性单因子IC


In [ ]:
# ============================================================
# 21. Descriptive Raw-Return Single-Factor IC
#
# This is a research diagnostic only.
# It is not used to select features using future OOS dates.
# ============================================================

raw_single_factor_rows = []

raw_daily_factor_ic_parts = []

for feature in raw_model_features:

    factor_part = (
        raw_return_master_df[
            [
                "date",
                "code",
                feature,
                RAW_MODEL_TARGET,
            ]
        ]
        .replace(
            [np.inf, -np.inf],
            np.nan,
        )
        .dropna()
        .copy()
    )

    if (
        len(factor_part) < 30
        or factor_part[
            feature
        ].nunique() < 2
    ):
        continue

    pooled_pearson_ic = (
        factor_part[
            feature
        ]
        .corr(
            factor_part[
                RAW_MODEL_TARGET
            ],
            method="pearson",
        )
    )

    pooled_rank_ic = (
        factor_part[
            feature
        ]
        .corr(
            factor_part[
                RAW_MODEL_TARGET
            ],
            method="spearman",
        )
    )

    daily_ic = (
        factor_part
        .groupby("date")
        .apply(
            lambda day: (
                day[feature]
                .corr(
                    day[
                        RAW_MODEL_TARGET
                    ],
                    method="spearman",
                )
                if (
                    day[feature].nunique() > 1
                    and day[
                        RAW_MODEL_TARGET
                    ].nunique() > 1
                )
                else np.nan
            ),
            include_groups=False,
        )
        .dropna()
    )

    raw_single_factor_rows.append({
        "feature":
            feature,

        "pooled_pearson_ic":
            pooled_pearson_ic,

        "pooled_rank_ic":
            pooled_rank_ic,

        "mean_daily_rank_ic":
            daily_ic.mean(),

        "median_daily_rank_ic":
            daily_ic.median(),

        "daily_ic_std":
            daily_ic.std(),

        "positive_daily_ic_ratio":
            (daily_ic > 0).mean(),

        "n_dates":
            len(daily_ic),

        "n_obs":
            len(factor_part),
    })

    if not daily_ic.empty:

        daily_ic_part = (
            daily_ic
            .rename("daily_rank_ic")
            .reset_index()
        )

        daily_ic_part[
            "feature"
        ] = feature

        raw_daily_factor_ic_parts.append(
            daily_ic_part
        )

raw_single_factor_summary = (
    pd.DataFrame(
        raw_single_factor_rows
    )
)

raw_single_factor_summary[
    "abs_mean_daily_rank_ic"
] = (
    raw_single_factor_summary[
        "mean_daily_rank_ic"
    ].abs()
)

raw_single_factor_summary = (
    raw_single_factor_summary
    .sort_values(
        "abs_mean_daily_rank_ic",
        ascending=False,
    )
    .reset_index(drop=True)
)

raw_daily_factor_ic = (
    pd.concat(
        raw_daily_factor_ic_parts,
        ignore_index=True,
    )
    if raw_daily_factor_ic_parts
    else pd.DataFrame()
)

display(
    raw_single_factor_summary.head(30)
)


In [ ]:
# ============================================================
# 21A. Top Raw-Return Single-Factor IC Plot
# ============================================================

RAW_TOP_IC_FEATURES = 20

raw_ic_plot = (
    raw_single_factor_summary
    .head(
        RAW_TOP_IC_FEATURES
    )
    .sort_values(
        "mean_daily_rank_ic"
    )
)

plt.figure(
    figsize=(10, 7)
)

colors = np.where(
    raw_ic_plot[
        "mean_daily_rank_ic"
    ] >= 0,
    "tab:blue",
    "tab:orange",
)

plt.barh(
    raw_ic_plot[
        "feature"
    ],
    raw_ic_plot[
        "mean_daily_rank_ic"
    ],
    color=colors,
)

plt.axvline(
    0,
    color="black",
    linewidth=1,
)

plt.xlabel(
    "Mean daily Spearman Rank IC"
)

plt.ylabel(
    "Feature"
)

plt.title(
    "Top Descriptive IC Features for Raw 5-Minute Return"
)

plt.grid(
    axis="x",
    alpha=0.3,
)

plt.tight_layout()
plt.show()


# Part VI. 因子深度研究框架

## 三棵重点因子树

### A. 严格 09:30 盘口压力

- 一、三、五、十档深度不平衡；
- 数量口径与金额口径；
- 等权与距离加权；
- microprice deviation；
- 竞价末盘口到严格开盘盘口的压力变化与持续性。

### B. 竞价缺口、吸收与反转

- 原始、市场残差和板块残差缺口；
- 正缺口、负缺口和绝对缺口；
- 缺口 × 开盘盘口支持；
- 缺口 × 大单支持；
- 高开回落、低开修复及 gap absorption。

### C. 大单与参与者结构

- 固定金额阈值；
- 股票内相对分位数阈值；
- 09:20 前后及最后 60/30 秒；
- 金额占比、笔数占比、买卖不平衡和集中度；
- 早晚方向反转、持续性、大单与小单分歧；
- 大单方向与最终盘口是否一致。


In [ ]:
# ============================================================
# Existing factor-family baseline
# ============================================================

factor_family_patterns = {
    "opening_book_pressure": [
        "strict_opening_", "final_depth_", "microprice",
    ],
    "gap_and_reversal": [
        "auction_gap", "auction_return", "residual_auction_gap",
        "gap_absorption", "auction_open_reversal",
    ],
    "large_order_structure": [
        "large_order", "large_flow", "large_vs_small",
        "late_large", "order_amount_hhi", "top5_order",
    ],
}

def assign_factor_family(feature):
    for family, patterns in factor_family_patterns.items():
        if any(pattern in feature for pattern in patterns):
            return family
    return "other"

factor_depth_baseline = raw_single_factor_summary.copy()
factor_depth_baseline["factor_family"] = (
    factor_depth_baseline["feature"].map(assign_factor_family)
)

display(
    factor_depth_baseline[
        factor_depth_baseline["factor_family"] != "other"
    ].sort_values(
        "abs_mean_daily_rank_ic",
        ascending=False,
    )
)


## 下一步：先扩展 Notebook 04 的大单因子工厂

当前 `large_order_imbalance` 使用固定 50 万元阈值，只能回答一个非常
有限的问题。下一次特征导出至少应加入：

| 维度 | 候选定义 |
|---|---|
| 固定金额 | 10万、30万、50万、100万、200万 |
| 相对规模 | 股票当日 P90、P95、P99；相对历史典型订单规模 |
| 时间窗口 | 09:15–09:20、09:20–09:24、09:24–09:25、最后60秒、最后30秒 |
| 方向 | 买入、卖出、净不平衡 |
| 结构 | 金额占比、笔数占比、集中度、Top-N占比 |
| 动态 | 持续、增强、衰减、买转卖、卖转买 |

阈值比较必须使用统一指标，并报告多重尝试数量。不能只展示全样本中
表现最好的阈值。


# Part VII. 评价和选择规则

每个母因子与裂变因子统一比较：

1. 覆盖率和唯一值数量；
2. mean/median daily Rank IC；
3. ICIR；
4. positive IC ratio；
5. Q1–Q5 五组收益与单调性；
6. Q5−Q1及正收益日期比例；
7. 前后子样本；
8. 高/中/低流动性；
9. 主板/创业板等板块；
10. 与母因子及同族因子的相关性。

每棵因子树最多保留 1–2 个经济含义清晰且稳定的变体，最终使用等权
rank 或仅由历史训练窗口决定的 IC 权重进行机械组合。


# Part VIII. 当前可交付结论

- 严格 09:30→09:35 标签已经建立并通过质量审计；
- 开盘五档深度不平衡和 microprice deviation 是当前最强的单因子；
- 竞价缺口的市场/板块残差及其绝对值具有反转/过度定价信息；
- 大单方向具有经济解释，但固定 50 万元定义的单体 IC 弱于盘口因子；
- 大单与小单差异、晚期大单变化、集中度和缺口支持关系值得继续深挖；
- 复杂模型能够产生一定横截面排序，但不能稳定预测绝对收益幅度；
- 因此下一阶段转向“少量重要母因子 → 机制裂变 → 稳健性比较”。
